# Trajectories, diversity, and corpus quality

A hands-on tour of the richer `pymc-generator`: **100 primary simulations**, with an eight-world baseline inspected first. All datasets are 104 weeks long; the retained baseline, atlas and replay examples bring the export manifest to **117 datasets**.

| Dataset set | Worlds | Purpose |
| --- | ---: | --- |
| Primary mixed shard | 100 (50 generation cells × 2 draws) | Parameter and zero-period distributions, full quality report, diversity and curation |
| Frozen mixed baseline | 8 (4 cells × 2 draws) | Run the new diagnostics before scaling; compare with the same definitions and shared bins |
| Trajectory atlas | 8 (one per archetype) | Recipes for launches, flights, ramps, shutdowns, doubling, seasonality, and trends |
| Replay world | 1 | Every component enabled; exact replay, reusable template, fixed-input oracle |

We generate and **save** eight worlds, inspect their parameter distributions and zero periods, and only then generate the 100-world primary shard. Both saved archives remain separate. You can analyze an existing file without regenerating it; every plot and table runs in the docs build.

**Read the units correctly.** Model inputs are `treatment_raw` (nonnegative channel sizes), `covariates` (signed controls), and `outcome_raw`. Normalized training tensors are separate. Component flags, schedules, coefficients, and causal contributions are **privileged generator truth**, not features to give a fitted model.

This implements the sparse example in [#29](https://github.com/pymc-labs/pymc-generator/issues/29) and the stored-corpus report in [#31](https://github.com/pymc-labs/pymc-generator/issues/31), using features [#24–#28](https://github.com/pymc-labs/pymc-generator/pull/38). This finite, four-count-combination showcase is not a representative training corpus.


## 1. Install and choose seeds

Use **Python 3.13+** and the checkout containing this notebook (branched from PR #38). Older released packages can share the version number while lacking these unmerged features; the checkout revision and locked dependencies matter.

```bash
uv sync --locked --extra docs
.venv/bin/python -m ipykernel install --prefix "$PWD/.venv" --name pymc-generator
# Open this notebook in your Jupyter-capable editor with that kernel, or execute it:
.venv/bin/python -m nbconvert --execute --to html \
  --ExecutePreprocessor.timeout=1800 --output-dir /tmp \
  docs/examples/trajectory-showcase.ipynb
```

No GPU, posterior sampling, or external data is needed. Set `PYMC_GENERATOR_SHOWCASE_DIR` before execution to keep outputs in a directory you choose; otherwise they go into a new temporary directory. The analysis/visual seed is independent of the generation seed.


In [ ]:
%matplotlib inline

import os
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc
import pytensor
from IPython.display import Image, display

import pymc_generator as pg
from pymc_generator import viz
from pymc_generator.presets import TRAJECTORY_ARCHETYPES
from pymc_generator.slots import TRAJECTORY_COMPONENTS, SlotLayout

GENERATION_SEED = 29
VISUAL_SEED = 31
VIF_THRESHOLD = 10.0
MAX_BAD_SHARE = 0.25

output_override = os.environ.get("PYMC_GENERATOR_SHOWCASE_DIR")
OUTPUT_DIR = (
    Path(output_override)
    if output_override
    else Path(tempfile.mkdtemp(prefix="pymc-generator-showcase-"))
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT_DIR))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print(
    "Versions:",
    {"pymc-generator": pg.__version__, "pymc": pymc.__version__, "pytensor": pytensor.__version__},
)
print("Kernel interpreter:", sys.executable)
print("Imported library:", pg.__file__)
print("Output directory:", RUN_DIR.resolve())

## 2. Generate a sparse, mixed-trajectory shard

Every world has **15 treatment slots and 10 control slots**, with 5–15 and 3–10 active inputs respectively. Activity masks distinguish padded slots from real zero-valued inputs. Exactly **five `C→Y` edges and three `Z→Y` edges** are present. All other edges are zero; the retained latent is disconnected.

We allocate one generation cell to each of four count combinations. All 88 combinations in the 11×8 grid remain visible in the report; 84 are deliberately given zero weight. To request uniform coverage of the whole grid, set `active_count_weights=None` and use at least 88 cells (outside this small tour).

`trajectories="composable"` gives each input independent inclusion flags for eight components: HF noise, pulses, onset, offset, flighting, level jumps, seasonal level, and trend. Flags are shared by draws from a cell; numerical schedules and series vary by draw. Realized prevalence need not equal the configured probability in four cells.

Mechanisms are **geometric carryover + Michaelis–Menten saturation**. The geometric kernel uses `pymc_marketing.geometric_adstock(normalize=True)`: weights sum to one, but inputs are **not** mapped to `[0, 1]`. We opt into log-uniform MM scales and reference-contribution priors; their anchors depend on parameters, never on the realized input mean.


In [ ]:
SPARSE_EDGES = dict.fromkeys(("cy", "dc", "dz", "dy", "zy", "zc", "cc", "zz"), (0, 0))
SPARSE_EDGES.update(cy=(5, 5), zy=(3, 3))
COUNT_COMBINATIONS = ((5, 3), (8, 5), (11, 7), (15, 10))
weights = np.zeros((11, 8))  # rows: 5..15 treatments; columns: 3..10 controls
for n_treatments, n_controls in COUNT_COMBINATIONS:
    weights[n_treatments - 5, n_controls - 3] = 1.0

shapes = pg.SCMPrior().saturation_prior_ranges
shapes["michaelis_menten"]["kappa_mult"] = (0.05, 20.0)
MECHANISMS = {
    "carryover_family_probs": {"none": 0.0, "geometric": 1.0, "weibull": 0.0},
    "saturation_family_probs": {
        name: float(name == "michaelis_menten")
        for name in ("linear", "hill", "logistic", "michaelis_menten", "tanh", "root")
    },
    "saturation_prior_ranges": shapes,
    "mm_scale_prior": "log_uniform",
    "treatment_reference_contribution_range": (0.5, 2.0),
    "covariate_reference_contribution_range": (-0.3, 0.3),
    "covariate_reference_scale": 2.0,
}
cfg = pg.make_scm_prior(
    n_treatments=15,
    n_covariates=10,
    n_latent=1,
    n_time_steps=104,
    n_treatments_active_range=(5, 15),
    n_covariates_active_range=(3, 10),
    n_cells=4,
    draws_per_cell=2,
    seed=GENERATION_SEED,
    edge_budget=SPARSE_EDGES,
    trajectories="composable",
    active_count_allocation="stratified",
    active_count_weights=weights.tolist(),
    **MECHANISMS,
)
generated = pg.sample_prior_predictive(cfg)
BASELINE_PATH = RUN_DIR / "baseline-8-worlds.npz"
CORPUS_PATH = BASELINE_PATH
pg.save_corpus(generated, CORPUS_PATH)
corpus = pg.load_corpus(CORPUS_PATH)
assert not pg.DataGenerator.validate_corpus(corpus)
for key, value in generated.items():
    if isinstance(value, np.ndarray):
        np.testing.assert_array_equal(corpus[key], value, err_msg=key)
    elif key == "identifiability":
        for name, array in value.items():
            np.testing.assert_array_equal(corpus[key][name], array, err_msg=name)
assert corpus["diagnostics"] == {
    key: value for key, value in generated["diagnostics"].items() if key != "timing"
}
print("Saved and loaded:", CORPUS_PATH.name, corpus["treatment_raw"].shape)
print(
    "Schema:",
    corpus["diagnostics"]["schema_version"],
    "| all numerical fields round-trip exactly; timing is not persisted",
)

In [ ]:
layout = SlotLayout(15, 10, 1)
edge_rows = []
for world_id in range(len(corpus["cell_id"])):
    graph = layout.unpack(corpus["g"][world_id])
    counts = {kind: int(block.sum()) for kind, block in graph.items()}
    assert counts == {kind: budget[0] for kind, budget in SPARSE_EDGES.items()}
    edge_rows.append(
        {
            "world_id": world_id,
            "cell_id": int(corpus["cell_id"][world_id]),
            "treatments": int(corpus["treatment_active_mask"][world_id].sum()),
            "controls": int(corpus["covariate_active_mask"][world_id].sum()),
            **counts,
        }
    )
assert corpus["treatment_raw"].shape == (8, 104, 15)
assert corpus["covariates"].shape == (8, 104, 10)
assert (corpus["treatment_raw"] >= 0).all()
assert (corpus["covariates"] < 0).any()  # signed controls are actually exercised
assert not corpus["latent_unobserved_contribution"].any()
assert not corpus["indirect_effects_by_source"].any()
for role, key in (("treatment", "treatment_raw"), ("covariate", "covariates")):
    active = corpus[f"{role}_active_mask"].astype(bool)
    assert not corpus[key].transpose(0, 2, 1)[~active].any()
    assert not corpus[key][corpus[f"{role}_activity"] == 0].any()
assert (corpus["carryover_family"][corpus["treatment_active_mask"] == 1] == 1).all()
assert (corpus["sat_family"][corpus["treatment_active_mask"] == 1] == 3).all()
display(pd.DataFrame(edge_rows).set_index("world_id"))

### Four worlds: inputs and outcome decomposition

One world per generation cell, not four repeated draws of one structure. Every active input is shown. A channel can be observed yet have no direct effect: activity is not the same as a `C→Y` edge. The decomposition includes signed control effects and observation noise; these are not input trajectories.


In [ ]:
def plot_world(c, world_id):
    """Plot one stored world; retain original IDs and omit only padding."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, layout="constrained")
    for ax, role, key, prefix in (
        (axes[0], "treatment", "treatment_raw", "C"),
        (axes[1], "covariate", "covariates", "Z"),
    ):
        slots = np.flatnonzero(c[f"{role}_active_mask"][world_id])
        for slot in slots:
            ax.plot(c[key][world_id, :, slot], linewidth=1, label=f"{prefix}{slot + 1}")
        ax.set_ylabel(f"{role} input units")
        ax.legend(ncol=5, fontsize=7, loc="upper right")
    parts = {
        "intrinsic baseline": c["baseline_intrinsic"][world_id],
        "direct treatments": c["treatment_contribution_raw"][world_id].sum(axis=1),
        "controls": c["covariate_contribution"][world_id].sum(axis=1),
        "latents": c["latent_unobserved_contribution"][world_id].sum(axis=1),
        "indirect": c["indirect_effects_by_source"][world_id].sum(axis=1),
        "observation noise": c["outcome_noise"][world_id],
    }
    reconstructed = sum(np.asarray(value, dtype=np.float64) for value in parts.values())
    np.testing.assert_allclose(reconstructed, c["outcome_raw"][world_id], rtol=3e-6, atol=3e-6)
    for label, values in parts.items():
        axes[2].plot(values, label=label, linewidth=1)
    axes[2].plot(c["outcome_raw"][world_id], color="black", label="outcome", linewidth=1.5)
    axes[2].set(xlabel="reported week (zero-based)", ylabel="outcome units")
    axes[2].legend(ncol=4, fontsize=7)
    fig.suptitle(
        f"World {world_id} · cell {int(c['cell_id'][world_id])} · decomposition max error {np.max(np.abs(reconstructed - c['outcome_raw'][world_id])):.2g}"
    )
    plt.show()
    plt.close(fig)

In [ ]:
_, first_per_cell = np.unique(corpus["cell_id"], return_index=True)
for world_id in first_per_cell:
    plot_world(corpus, int(world_id))

## 3. Measure trajectories, not just component flags

Two distinct questions:

* **What was configured and drawn?** Stored flags and exact schedule leaves answer this. A seasonal flag can coexist with flighting and a negative trend.
* **What does the model actually receive?** Measure `treatment_raw` / `covariates`, including noise and gates, rather than the outcome contribution after carryover/saturation.

Below each active input gets zero fraction, first/last nonzero week, off→on/on→off transitions, and on-run count. Trend slopes use **nonzero weeks**, time scaled to `[0, 1]`, and input divided by its RMS over those weeks. Signed slope tracks the input value; magnitude slope tracks `abs(input)` (so a negative control approaching zero is decaying in magnitude). Fewer than two nonzero weeks yields an undefined slope, not zero.

The visible-shape screen calls a delayed input a **ramp after launch** if its magnitude slope is at least `0.25`, and a stopped input a **decay to shutdown** if it is at most `-0.25`. These are configurable descriptive heuristics, **not** monotonicity proofs, component classifiers, or statistical tests. An exact doubling *schedule* means a stored treatment jump factor equals two, not that two noisy neighboring observations have ratio two.


In [ ]:
def normalized_slopes(x, on):
    indexes = np.flatnonzero(on)
    if indexes.size < 2:
        return np.nan, np.nan
    values = np.asarray(x[indexes], dtype=np.float64)
    values = values / np.max(np.abs(values))
    values = values / np.sqrt(np.mean(values**2))
    time = indexes / max(len(x) - 1, 1)
    time = time - time.mean()
    denominator = time @ time
    return float(time @ (values - values.mean()) / denominator), float(
        time @ (np.abs(values) - np.abs(values).mean()) / denominator
    )


def measure_inputs(c, *, slope_threshold=0.25):
    records = []
    has_truth = "treatment_components" in c and "covariate_components" in c
    for role, key, prefix in (
        ("treatment", "treatment_raw", "C"),
        ("covariate", "covariates", "Z"),
    ):
        for world_id, slot in np.argwhere(c[f"{role}_active_mask"] == 1):
            x = c[key][world_id, :, slot]
            on = x != 0.0
            indexes = np.flatnonzero(on)
            signed_slope, magnitude_slope = normalized_slopes(x, on)
            launches = int(np.count_nonzero(~on[:-1] & on[1:]))
            stops = int(np.count_nonzero(on[:-1] & ~on[1:]))
            row = {
                "world_id": int(world_id),
                "role": role,
                "slot": int(slot),
                "input": f"{prefix}{slot + 1}",
                "zero_fraction": float((~on).mean()),
                "first_nonzero_week": int(indexes[0]) if indexes.size else None,
                "last_nonzero_week": int(indexes[-1]) if indexes.size else None,
                "off_to_on": launches,
                "on_to_off": stops,
                "on_runs": launches + int(on[0]),
                "signed_slope": signed_slope,
                "magnitude_slope": magnitude_slope,
            }
            row.update(
                always_on=bool(on.all()),
                all_zero=bool(not on.any()),
                delayed_start=bool(indexes.size and indexes[0] > 0),
                terminal_shutdown=bool(indexes.size and indexes[-1] < len(x) - 1),
                off_on_again=bool(row["on_runs"] >= 2),
            )
            row["ramp_after_launch"] = row["delayed_start"] and magnitude_slope >= slope_threshold
            row["decay_to_shutdown"] = (
                row["terminal_shutdown"] and magnitude_slope <= -slope_threshold
            )
            if has_truth:
                flags = c[f"{role}_components"][world_id, slot].astype(bool)
                row.update(
                    {
                        f"carries_{name}": bool(flag)
                        for name, flag in zip(TRAJECTORY_COMPONENTS, flags)
                    }
                )
                row["schedule_trend_change"] = (
                    float(c[f"trajectory_{role}_trend_change"][world_id, slot])
                    if row["carries_trend"]
                    else 0.0
                )
                row["seasonal_period_weeks"] = (
                    float(c[f"trajectory_{role}_seasonal_period"][world_id, slot])
                    if row["carries_seasonal"]
                    else np.nan
                )
                row["exact_doubling_schedule"] = bool(
                    role == "treatment"
                    and row["carries_level_jump"]
                    and (
                        c["trajectory_treatment_level_jump_factor"][world_id, :, slot] == 2.0
                    ).any()
                )
            records.append(row)
    return pd.DataFrame.from_records(records)


OBSERVED_SHAPES = (
    "always_on",
    "all_zero",
    "delayed_start",
    "terminal_shutdown",
    "off_on_again",
    "ramp_after_launch",
    "decay_to_shutdown",
)


def trajectory_inventory(measurements):
    rows = []
    for role in ("treatment", "covariate"):
        group = measurements[measurements["role"] == role]
        for name in OBSERVED_SHAPES:
            count = int(group[name].sum())
            rows.append(
                {
                    "role": role,
                    "trajectory": name,
                    "evidence": "reported input",
                    "count": count,
                    "eligible_inputs": len(group),
                    "status": "observed" if count else "absent in this shard",
                }
            )
        if role == "treatment" and "exact_doubling_schedule" in group:
            count = int(group["exact_doubling_schedule"].sum())
            rows.append(
                {
                    "role": role,
                    "trajectory": "exact_doubling_schedule",
                    "evidence": "stored schedule truth",
                    "count": count,
                    "eligible_inputs": len(group),
                    "status": "observed" if count else "absent in this shard",
                }
            )
    return pd.DataFrame(rows)

In [ ]:
# Boundary examples make the observable measurement definitions inspectable.
boundary_cases = {
    "all_zero": [0, 0, 0, 0, 0],
    "first_week_only": [2, 0, 0, 0, 0],
    "last_week_only": [0, 0, 0, 0, 2],
    "negative_constant": [-2, -2, -2, -2, -2],
    "zero_signed_mean": [-1, 0, 1],
    "approaching_zero": [-3, -2, -1],
    "gapped_rise": [0, 1, 0, 0, 4],
    "three_flights": [1, 0, 1, 0, 1],
}
boundary_rows = []
for label, values in boundary_cases.items():
    x = np.asarray(values, dtype=np.float64)
    view = {
        "treatment_raw": np.empty((1, len(x), 0)),
        "treatment_active_mask": np.empty((1, 0), dtype=np.uint8),
        "covariates": x[None, :, None],
        "covariate_active_mask": np.ones((1, 1), dtype=np.uint8),
    }
    row = measure_inputs(view).iloc[0]
    boundary_rows.append(row.to_dict() | {"example": label})
boundaries = pd.DataFrame(boundary_rows).set_index("example")
assert boundaries.loc["all_zero", "on_runs"] == 0
assert pd.isna(boundaries.loc["all_zero", "first_nonzero_week"])
assert boundaries.loc["first_week_only", "off_to_on"] == 0
assert boundaries.loc["first_week_only", "on_to_off"] == 1
assert boundaries.loc["last_week_only", "off_to_on"] == 1
assert boundaries.loc["last_week_only", "on_to_off"] == 0
assert boundaries.loc["negative_constant", "on_runs"] == 1
np.testing.assert_array_equal(
    boundaries.loc["negative_constant", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    0.0,
)
assert (
    boundaries.loc[["all_zero", "first_week_only", "last_week_only"], "signed_slope"].isna().all()
)
np.testing.assert_allclose(
    boundaries.loc["zero_signed_mean", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    [2.0, 0.0],
)
np.testing.assert_allclose(
    boundaries.loc["approaching_zero", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    [2 / np.sqrt(14 / 3), -2 / np.sqrt(14 / 3)],
)
np.testing.assert_allclose(
    boundaries.loc["gapped_rise", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    4 / np.sqrt(8.5),
)
assert boundaries.loc["three_flights", "on_runs"] == 3
assert boundaries.loc["three_flights", "off_to_on"] == 2
assert boundaries.loc["three_flights", "on_to_off"] == 2
display(
    boundaries[
        [
            "first_nonzero_week",
            "last_nonzero_week",
            "off_to_on",
            "on_to_off",
            "on_runs",
            "signed_slope",
            "magnitude_slope",
        ]
    ].round(3)
)

In [ ]:
measurements = measure_inputs(corpus)
display(
    measurements[
        [
            "world_id",
            "role",
            "input",
            "zero_fraction",
            "off_to_on",
            "on_to_off",
            "on_runs",
            "signed_slope",
            "magnitude_slope",
            "schedule_trend_change",
        ]
    ]
    .head(16)
    .round(3)
)
display(trajectory_inventory(measurements))

## 4. A deterministic quality report from the file

`quality_report(path, ...)` below does **no generation** and does not depend on `cfg`. It reloads the archive, validates it, recalculates prevalence and coverage from masks, computes descriptors and within-world diagnostics, and selects seeded visual examples.

**Prevalence denominator:** active `(world, input)` pairs, never padded slots or off-weeks. This matches the stored diagnostics. Eight worlds represent **four independent generation structures**, so repeated flags across a cell's draws are not eight independent component draws.

**Empirical identifiability screen:** observed active `C + Z` design, centered and unit-column-normalized by the library. A world is flagged if its numerical rank is below the active predictor count **or** any valid predictor VIF is at least `VIF_THRESHOLD` (infinite VIF counts). Constants reduce rank; invalid or absent designs are explicitly unassessed, never automatic passes. Read **differences first**, with levels as a companion: independent random walks can have high level VIF. Condition numbers are also reported, but have no extra threshold here.

The rank uses the library's SVD tolerance `max(eps64, eps64 × max(n_observations, n_varying_columns) × largest_singular_value)`. VIF is computed by residual projection onto the other predictors, not inferred from rank. The default threshold `10` is a notebook policy, **not a library default**. The acceptable flagged share defaults to `25%` and must remain below `50%` if the policy is to require a minority.

This is a finite-sample redundancy screen of raw inputs, **not proof of causal identifiability, nonlinear-response parameter recovery, or training utility**. Quality PASS/FAIL is reported honestly and never filters worlds away.

For **report-only use**, set `PYMC_GENERATOR_REPORT_PATH` to an existing shard and execute all cells tagged `report-setup`, `report-definitions`, or `report-run`, in notebook order. Skip generation and atlas cells; the tagged path includes the shared plotting helpers and seeded input/world visual checks. The [corpus-quality guide](https://pymc-labs.github.io/pymc-generator/guide/corpus-quality/) also gives a fresh-kernel command. PASS means a completely assessed, nonempty selection with flagged share **≤** the printed limit.


In [ ]:
from collections.abc import Mapping

from pymc_generator.sampler import CARRYOVER_FAMILY_KEYS, SATURATION_FAMILY_KEYS
from pymc_generator.slots import (
    CORPUS_ARRAY_FIELDS,
    MECHANISM_ARRAY_FIELDS,
    MECHANISM_PRIOR_FIELDS,
    MECHANISM_REFERENCE_FIELDS,
    PRIOR_COND_LAYOUT,
    TRAJECTORY_PARAM_FIELDS,
)


def _parameter_table(rows, table):
    schemas = {
        "values": {
            "parameter": "string",
            "role": "string",
            "world_id": "int64",
            "cell_id": "int64",
            "slot": "int64",
            "event": "int64",
            "value": "float64",
            "weight": "float64",
            "support_low": "float64",
            "support_high": "float64",
            "support_low_tolerance": "float64",
            "support_high_tolerance": "float64",
        },
        "catalog": {
            "parameter": "string",
            "field": "string",
            "role": "string",
            "group": "string",
            "label": "string",
            "units": "string",
            "kind": "string",
            "law": "string",
            "support_source": "string",
            "low": "float64",
            "high": "float64",
            "configured_fixed": "bool",
            "configured_probability": "float64",
            "categories": "string",
            "category_labels": "string",
            "sampling_unit": "string",
            "eligibility": "string",
            "availability": "string",
            "n_stored": "int64",
            "n_active": "int64",
            "n_eligible": "int64",
            "n_invalid": "int64",
            "padded_count": "int64",
            "drawn_but_unused_count": "int64",
            "unselected_component_count": "int64",
            "n_values": "int64",
            "n_worlds": "int64",
            "n_cells": "int64",
            "n_unique": "int64",
            "observed_min": "float64",
            "observed_max": "float64",
            "observed_constant": "bool",
            "dtype": "string",
            "storage_precision": "string",
            "note": "string",
        },
        "bins": {
            "cohort": "string",
            "parameter": "string",
            "role": "string",
            "bin": "int64",
            "left": "float64",
            "right": "float64",
            "bin_kind": "string",
            "bin_basis": "string",
            "count": "int64",
            "mass": "float64",
            "expected_mass": "float64",
        },
        "summary": {
            "cohort": "string",
            "parameter": "string",
            "role": "string",
            "label": "string",
            "units": "string",
            "availability": "string",
            "kind": "string",
            "law": "string",
            "support_source": "string",
            "low": "float64",
            "high": "float64",
            "sampling_unit": "string",
            "bin_basis": "string",
            "n_values": "int64",
            "n_worlds": "int64",
            "n_cells": "int64",
            "n_invalid": "int64",
            "drawn_but_unused_count": "int64",
            "unselected_component_count": "int64",
            "n_unique": "int64",
            "observed_min": "float64",
            "observed_max": "float64",
            "observed_constant": "bool",
            "configured_fixed": "bool",
            "n_bins": "int64",
            "occupied_bins": "Int64",
            "empty_bins": "Int64",
            "occupied_bin_fraction": "float64",
            "max_bin_mass": "float64",
            "max_abs_expected_deviation": "float64",
            "total_variation_from_expected": "float64",
            "n_below_support": "Int64",
            "n_above_support": "Int64",
            "n_outside_support": "Int64",
            "outside_support_mass": "float64",
            "n_below_shared_bins": "int64",
            "n_above_shared_bins": "int64",
            "reference_note": "string",
            "coverage_limitation": "string",
        },
    }
    schema = schemas[table]
    return pd.DataFrame.from_records(rows, columns=list(schema)).astype(schema)


def _parameter_range(bounds):
    if bounds is None:
        return None
    values = np.asarray(bounds, dtype=np.float64)
    if values.shape != (2,) or not np.isfinite(values).all() or values[0] > values[1]:
        return None
    return float(values[0]), float(values[1])


def _parameter_specs(c):
    """Describe archive fields without substituting the current SCMPrior defaults."""
    diagnostics = c.get("diagnostics", {})
    mechanisms = diagnostics.get("mechanism_priors", {})
    conditioning = diagnostics.get("prior_cond", {})
    trajectory = diagnostics.get("trajectory", {})
    specs = []

    def add(parameter, role, group, label, units, *, kind="sampled_or_fixed", **extra):
        spec = {
            "parameter": parameter,
            "field": parameter,
            "role": role,
            "group": group,
            "label": label,
            "units": units,
            "kind": kind,
            "law": "unknown",
            "support_source": "configured bounds/law not persisted",
            "low": np.nan,
            "high": np.nan,
            "configured_fixed": False,
            "configured_probability": np.nan,
            "categories": "",
            "category_labels": "",
            "sampling_unit": "accepted_world",
            "eligibility": "active input slots; exact zero values retained",
            "note": "Accepted worlds are conditioned on the generator's realism/overflow filters.",
        }
        spec.update(extra)
        specs.append(spec)
        return spec

    def recorded_range(spec, bounds, source, law="uniform"):
        interval = _parameter_range(bounds)
        if interval is not None:
            spec.update(low=interval[0], high=interval[1], support_source=source, law=law)
            if interval[0] == interval[1]:
                spec.update(configured_fixed=True, law="point_mass")
                if spec["kind"] != "derived":
                    spec["kind"] = "fixed"

    for field, keys, label in (
        ("carryover_family", CARRYOVER_FAMILY_KEYS, "carryover family"),
        ("sat_family", SATURATION_FAMILY_KEYS, "response family"),
    ):
        add(
            field,
            "treatment",
            "Structural response choices",
            label,
            "family ID",
            kind="categorical",
            law="categorical_unknown_probabilities",
            categories=",".join(str(i) for i in range(len(keys))),
            category_labels="; ".join(f"{i}={key}" for i, key in enumerate(keys)),
            sampling_unit="generation_cell",
            note=(
                "Family probabilities are not persisted. IDs are schema categories, not prior "
                "support coverage. Repeated worlds in a cell repeat its structural draw; "
                "n_cells, not n_values or n_worlds, counts structure units."
            ),
        )

    for field, family, label, units in (
        ("carryover_alpha", "geometric", "geometric carryover decay", "decay fraction"),
        ("weibull_lam", "weibull", "Weibull carryover scale", "weeks"),
        ("weibull_k", "weibull", "Weibull carryover shape", "shape"),
    ):
        spec = add(
            field,
            "treatment",
            "Carryover parameters",
            label,
            units,
            law="uniform_or_fixed_unknown_support",
            family_field="carryover_family",
            family_id=CARRYOVER_FAMILY_KEYS.index(family),
            eligibility=f"active treatments AND carryover_family={family}",
            note=(
                "Other active families have genuinely drawn, but unused, shape values; they "
                "are counted separately, not confused with inactive padding."
            ),
        )
        if field == "carryover_alpha" and conditioning:
            spec["conditioning_quantity"] = "carryover_alpha"
            recorded_range(
                spec,
                conditioning.get("supports", {}).get("carryover_alpha"),
                "diagnostics.prior_cond.supports.carryover_alpha + stored prior_cond intervals",
                "conditional_uniform",
            )

    for field, (family, leaf) in MECHANISM_PRIOR_FIELDS.items():
        spec = add(
            field,
            "treatment",
            "Response shape parameters",
            f"{family.replace('_', ' ')} {leaf.replace('_', ' ')}",
            "relative anchor multiplier" if leaf == "kappa_mult" else "shape",
            family_field="sat_family",
            family_id=SATURATION_FAMILY_KEYS.index(family),
            eligibility=f"active treatments AND sat_family={family}",
            note=(
                "Shape values for unselected response families were drawn but are not used. "
                "The selected physical response is described before its direct-edge gate."
            ),
        )
        law = "uniform"
        if field == "mm_kappa_mult":
            law = mechanisms.get("mm_scale_prior", "unknown")
        recorded_range(
            spec,
            mechanisms.get("saturation_prior_ranges", {}).get(family, {}).get(leaf),
            f"diagnostics.mechanism_priors.saturation_prior_ranges.{family}.{leaf}",
            law,
        )
        if field == "hill_slope" and conditioning:
            spec["conditioning_quantity"] = "hill_shape"
            spec["configured_fixed"] = False
            recorded_range(
                spec,
                conditioning.get("supports", {}).get("hill_shape"),
                "diagnostics.prior_cond.supports.hill_shape + stored prior_cond intervals",
                "conditional_uniform",
            )

    for field, label, units, role in (
        ("beta", "treatment outcome loading", "outcome / physical response", "treatment"),
        ("rho_zy", "signed covariate outcome loading", "outcome / covariate unit", "covariate"),
        ("treatment_level", "treatment drive anchor", "raw treatment units", "treatment"),
        ("saturation_scale", "legacy response anchor", "raw treatment units", "treatment"),
        ("mechanism_saturation_scale", "exact response anchor", "raw treatment units", "treatment"),
        ("treatment_means", "reported treatment mean", "raw treatment units", "treatment"),
        ("outcome_scale", "outcome normalization scale", "raw outcome units", "world"),
        ("confounding_strength", "innovation correlation strength", "correlation", "world"),
    ):
        derived = field not in ("beta", "rho_zy", "confounding_strength")
        spec = add(
            field,
            role,
            "Response loadings and anchors",
            label,
            units,
            kind="derived" if derived else "sampled_or_fixed",
            law="derived" if derived else "unknown",
            eligibility="all stored worlds" if role == "world" else "active input slots",
        )
        if field in ("beta", "rho_zy"):
            spec["note"] = (
                "Pre-edge coefficient, including direct-null active inputs; it is not the "
                "edge-gated coefficient or a realized time-series contribution."
            )
        if field == "beta" and mechanisms.get("treatment_reference_contribution_range") is not None:
            spec.update(kind="derived", law="derived_nonuniform")
            spec["note"] += " Derived target / nonlinear reference response; no uniform promise."
        if (
            field == "rho_zy"
            and mechanisms.get("covariate_reference_contribution_range") is not None
        ):
            spec.update(kind="derived", law="derived_unknown_support")
            target = _parameter_range(mechanisms.get("covariate_reference_contribution_range"))
            scale = mechanisms.get("covariate_reference_scale")
            if target is not None and scale is not None and np.isfinite(scale) and float(scale) > 0:
                recorded_range(
                    spec,
                    [target[0] / float(scale), target[1] / float(scale)],
                    "diagnostics.mechanism_priors.covariate_reference_contribution_range / covariate_reference_scale",
                )
                spec["note"] += (
                    " A uniform target divided by a fixed positive scale remains uniform."
                )
        if field in ("saturation_scale", "mechanism_saturation_scale"):
            spec["note"] += (
                " Same derived anchor, retained at legacy float32 and exact float64 precision."
            )
        if field == "confounding_strength":
            spec["note"] += (
                " A constant observed value does not establish a configured fixed prior."
            )

    for field in MECHANISM_REFERENCE_FIELDS:
        role = field.split("_", 1)[0]
        leaf = field.removeprefix(f"{role}_reference_")
        spec = add(
            field,
            role,
            "Reference contribution parameters",
            f"{role} reference {leaf.replace('_', ' ')}",
            "outcome units" if leaf == "contribution" else f"{role} / response units",
            kind="sampled_or_fixed" if leaf == "contribution" else "derived",
            law="unknown" if leaf == "contribution" else "derived",
            note="Parameter-only reference quantity before the direct-edge gate, not a realized effect.",
        )
        if leaf == "contribution":
            recorded_range(
                spec,
                mechanisms.get(f"{role}_reference_contribution_range"),
                f"diagnostics.mechanism_priors.{role}_reference_contribution_range",
            )
        elif field == "covariate_reference_input":
            scale = mechanisms.get("covariate_reference_scale")
            if scale is not None and np.isfinite(scale):
                recorded_range(
                    spec,
                    [float(scale), float(scale)],
                    "diagnostics.mechanism_priors.covariate_reference_scale",
                )
                spec["kind"] = "fixed"
        elif leaf == "response":
            spec["note"] += " Derived from the selected saturation family and reference input."
        else:
            spec["note"] += " Derived response anchor times the recorded reference multiplier."

    for role in ("treatment", "covariate"):
        for component in TRAJECTORY_COMPONENTS:
            probability = trajectory.get("inclusion_probs", {}).get(role, {}).get(component)
            add(
                f"{role}_components_{component}",
                role,
                f"Structural {role} components",
                f"{role} carries {component}",
                "0/1 inclusion flag",
                field=f"{role}_components",
                component_column=TRAJECTORY_COMPONENTS.index(component),
                kind="structural_flag",
                law="bernoulli" if probability is not None else "bernoulli_unknown_probability",
                support_source=(
                    f"diagnostics.trajectory.inclusion_probs.{role}.{component}"
                    if probability is not None
                    else "inclusion probability not persisted"
                ),
                configured_probability=float(probability) if probability is not None else np.nan,
                low=(float(probability) if probability in (0, 1) else 0.0)
                if probability is not None
                else np.nan,
                high=(float(probability) if probability in (0, 1) else 1.0)
                if probability is not None
                else np.nan,
                configured_fixed=probability in (0, 1),
                categories="0,1",
                category_labels="0=not carried; 1=carried",
                sampling_unit="generation_cell",
                note=(
                    "False and true flags are both meaningful active-input observations. Worlds "
                    "within one cell repeat these flags. This per-world descriptive plot has no "
                    "independent-draw prior fit score; use n_cells for structure units."
                ),
            )

    for field, (axes, dtype) in TRAJECTORY_PARAM_FIELDS.items():
        suffix = field.removeprefix("trajectory_")
        role, suffix = suffix.split("_", 1)
        component = next(name for name in TRAJECTORY_COMPONENTS if suffix.startswith(f"{name}_"))
        leaf = suffix.removeprefix(f"{component}_")
        discrete = np.issubdtype(dtype, np.integer)
        units = "weeks" if leaf in ("start", "stop", "week", "period", "on_weeks") else "magnitude"
        if (component, leaf) in (("hf", "sigma"), ("pulse", "amp")):
            units = "raw treatment-drive units" if role == "treatment" else "raw covariate units"
        if (component, leaf) == ("seasonal", "amplitude"):
            units = "log-level amplitude" if role == "treatment" else "raw covariate units"
        if (component, leaf) == ("level_jump", "size"):
            units = "raw covariate units"
        if leaf == "phase":
            units = "weeks" if component == "flighting" else "radians"
        if component == "trend" or leaf == "log_factor":
            units = "log-level change" if role == "treatment" else "raw covariate units"
        if leaf == "factor":
            units = "level multiplier"
        if leaf == "prob":
            units = "probability"
        law = "uniform_or_fixed_unknown_support"
        kind = "discrete" if discrete else "sampled_or_fixed"
        note = (
            "Configured trajectory recipe bounds are not persisted; no default bounds are assumed."
        )
        if (component, leaf) in (("hf", "sigma"), ("pulse", "amp")):
            kind, law = "derived", "derived_nonuniform"
            note += (
                " Effective magnitude: primitive multiplier times the treatment drive anchor"
                " or covariate walk std. The multiplier is not separately persisted;"
                " varying random scales remove any uniform marginal promise."
            )
        if component == "level_jump" and leaf == "factor":
            law = "log_uniform_or_fixed_unknown_support"
        if (component, leaf) in (("onset", "start"), ("offset", "stop"), ("level_jump", "week")):
            kind, law = "derived_discrete", "derived_discrete_unknown_support"
            note += " Integer timing is derived from fractional/stratified draws, not a continuous uniform."
        elif component == "flighting":
            law = (
                "discrete_uniform_or_fixed_unknown_support"
                if leaf == "period"
                else "derived_discrete"
            )
            kind = "discrete" if leaf == "period" else "derived_discrete"
            note += (
                " Duty and phase primitives are not persisted; on_weeks/phase depend on the period."
            )
        elif component == "level_jump" and leaf == "log_factor":
            note += (
                " Stored log of the factor; its primitive is uniform in log-space when not fixed."
            )
        elif (component, leaf) == ("seasonal", "period"):
            note += " Even an observed constant 52 does not prove a configured point mass."
        elif (component, leaf) == ("seasonal", "phase"):
            note += " The source algorithm uses a fixed phase law, but no archive-recorded support is supplied."
        add(
            field,
            role,
            f"Trajectory {role}: {component}",
            f"{role} {component.replace('_', ' ')} {leaf.replace('_', ' ')}",
            units,
            component=component,
            kind=kind,
            law=law,
            eligibility=f"active {role} inputs AND carries {component}; all event axes retained",
            note=note,
        )

    for field, kind, label, units in (
        ("treatment_shock_index", "discrete", "shock treatment slot", "zero-based treatment slot"),
        ("treatment_shock_start", "derived_discrete", "shock start", "weeks"),
        ("treatment_shock_length", "discrete", "shock length", "weeks"),
        (
            "treatment_shock_level_multiplier",
            "sampled_or_fixed",
            "shock level multiplier",
            "level multiplier",
        ),
        ("treatment_shock_level", "derived", "shock absolute level", "raw treatment units"),
    ):
        add(
            field,
            "treatment",
            "Legacy treatment shock parameters",
            label,
            units,
            kind=kind,
            law="derived" if kind.startswith("derived") else "unknown",
            shock=True,
            eligibility="stored shock events targeting active treatment slots",
            note="Zero configured events are an empty event population, not zero-valued sampled shocks.",
        )

    for column in PRIOR_COND_LAYOUT:
        quantity, part = column.rsplit("_", 1)
        spec = add(
            f"prior_cond_{column}",
            "world",
            "Cell-level conditioning intervals",
            f"{quantity.replace('_', ' ')} interval {part}",
            "parameter units",
            field="prior_cond",
            prior_column=PRIOR_COND_LAYOUT.index(column),
            kind="conditioning_interval",
            law="conditional_uniform" if part == "low" else "uniform",
            sampling_unit="generation_cell",
            eligibility="stored cell-level prior_cond rows; repeated worlds are not independent intervals",
            note=(
                "Interval lows depend on the drawn width. The per-world rows repeat a cell's "
                "interval; no independent per-world prior fit overlay is drawn."
            ),
        )
        if part == "width":
            recorded_range(
                spec,
                conditioning.get("width_ranges", {}).get(quantity),
                f"diagnostics.prior_cond.width_ranges.{quantity}",
            )

    for field, role, label, units in (
        (
            "treatment_reference_multiplier",
            "treatment",
            "configured treatment reference multiplier",
            "multiplier",
        ),
        (
            "covariate_reference_scale",
            "covariate",
            "configured covariate reference scale",
            "covariate units",
        ),
    ):
        spec = add(
            field,
            role,
            "Reference recipe metadata",
            label,
            units,
            metadata_only=True,
            kind="configuration",
            sampling_unit="configuration",
            eligibility="configuration metadata only; not an empirical population",
            note="One recorded setting, not one independently sampled value per world or input.",
        )
        value = mechanisms.get(field)
        if value is not None and np.isfinite(value):
            recorded_range(
                spec, [float(value), float(value)], f"diagnostics.mechanism_priors.{field}"
            )

    for prefix, role in (
        ("rw_c", "treatment"),
        ("rw_z", "covariate"),
        ("rw_d", "latent"),
        ("rw_b", "baseline"),
        ("rw_y", "outcome"),
    ):
        for leaf in ("mean", "std"):
            add(
                f"{prefix}_{leaf}",
                role,
                "Unpersisted baseline, walk and noise parameters",
                f"{role} walk/noise {leaf}",
                "raw process units",
                unpersisted=True,
                kind="unavailable",
                sampling_unit="unpersisted",
                eligibility="unavailable from a saved corpus",
                note="Walk/noise primitives are not stored; realized paths cannot reconstruct these draws or bounds.",
            )
    for role, suffix in (
        ("treatment", "c"),
        ("covariate", "z"),
        ("latent", "d"),
        ("baseline", "b"),
    ):
        add(
            f"smoothness_{suffix}",
            role,
            "Unpersisted baseline, walk and noise parameters",
            f"{role} walk smoothness",
            "smoothness",
            unpersisted=True,
            kind="unavailable",
            sampling_unit="unpersisted",
            eligibility="unavailable from a saved corpus",
            note="The generation-cell smoothness draw and its configured law are not persisted.",
        )
    for field, edge in (
        ("w_dc", "D→C"),
        ("u_dz", "D→Z"),
        ("v_zc", "Z→C"),
        ("alpha_cc", "C→C"),
        ("gamma_zz", "Z→Z"),
        ("delta_dy", "D→Y"),
    ):
        add(
            field,
            "edge",
            "Unpersisted indirect and latent-edge coefficients",
            f"{edge} coefficient",
            "destination / source units",
            unpersisted=True,
            kind="unavailable",
            sampling_unit="unpersisted",
            eligibility="unavailable from a saved corpus",
            note="Graph existence and aggregate indirect effects do not reconstruct individual edge coefficients.",
        )
    for role in ("treatment", "covariate"):
        add(
            f"trajectory_{role}_flighting_duty",
            role,
            f"Trajectory {role}: flighting",
            f"{role} flighting duty primitive",
            "fraction",
            unpersisted=True,
            kind="unavailable",
            sampling_unit="unpersisted",
            eligibility="unavailable from a saved corpus",
            note="Only rounded on_weeks/period are persisted; they do not recover the duty draw.",
        )
    return specs


def _parameter_precision(dtype):
    if np.issubdtype(dtype, np.floating):
        return f"{dtype.name}; {np.finfo(dtype).nmant + 1} significant binary bits"
    return f"{dtype.name}; exact stored integers"


def parameter_report(c):
    """Catalog saved parameter truth and finite active/family/component-masked values.

    Each parameter gives every world with finite applicable values equal total
    weight, then divides that weight among its slots/events. A missing draw is
    not imputed. world_id is the archive row; cell_id retains generation identity.
    slot/event are zero-based, with -1 meaning no slot/event axis. Structural
    rows are explicitly repeated cell-level draws, never independent-world fits.
    """
    cell_ids = np.asarray(c["cell_id"], dtype=np.int64)
    fields = {
        **CORPUS_ARRAY_FIELDS,
        **MECHANISM_ARRAY_FIELDS,
        **MECHANISM_REFERENCE_FIELDS,
        **TRAJECTORY_PARAM_FIELDS,
    }
    catalog_rows, value_rows = [], []
    catalog_columns = set(_parameter_table([], "catalog").columns)
    for spec in _parameter_specs(c):
        row = {key: value for key, value in spec.items() if key in catalog_columns}
        row.update(
            availability="unavailable",
            n_stored=0,
            n_active=0,
            n_eligible=0,
            n_invalid=0,
            padded_count=0,
            drawn_but_unused_count=0,
            unselected_component_count=0,
            n_values=0,
            n_worlds=0,
            n_cells=0,
            n_unique=0,
            observed_min=np.nan,
            observed_max=np.nan,
            observed_constant=False,
            dtype="unavailable",
            storage_precision="unavailable",
        )
        field = spec["field"]
        if spec.get("metadata_only"):
            if np.isfinite(spec["low"]):
                row.update(
                    availability="metadata_only",
                    dtype="float64",
                    storage_precision="recorded scalar configuration",
                )
            catalog_rows.append(row)
            continue
        if spec.get("unpersisted") or field not in c:
            if field in fields:
                row["note"] += " This stored truth field is absent from this archive."
            catalog_rows.append(row)
            continue
        array = np.asarray(c[field])
        if "component_column" in spec:
            array = array[:, :, spec["component_column"]]
        elif "prior_column" in spec:
            array = array[:, spec["prior_column"]]
        row.update(
            n_stored=array.size,
            dtype=array.dtype.name,
            storage_precision=_parameter_precision(array.dtype),
        )
        role = spec["role"]
        if spec.get("shock"):
            targets = np.asarray(c["treatment_shock_index"], dtype=np.int64)
            mask = np.asarray(c["treatment_active_mask"]) == 1
            valid = (targets >= 0) & (targets < mask.shape[1])
            active = np.zeros(targets.shape, dtype=bool)
            worlds, events = np.nonzero(valid)
            active[worlds, events] = mask[worlds, targets[worlds, events]]
        elif role == "world":
            active = np.ones(array.shape, dtype=bool)
        else:
            active = np.asarray(c[f"{role}_active_mask"]) == 1
            if array.ndim == 3:
                active = np.broadcast_to(active[:, None, :], array.shape)
        row.update(n_active=int(active.sum()), padded_count=int(array.size - active.sum()))
        eligible = active
        if "family_field" in spec:
            if spec["family_field"] not in c:
                row["availability"] = "eligibility_unavailable"
                row["note"] += (
                    " Selection-family truth is absent; values cannot be classified as live."
                )
                catalog_rows.append(row)
                continue
            eligible = active & (np.asarray(c[spec["family_field"]]) == spec["family_id"])
            row["drawn_but_unused_count"] = int((active & ~eligible).sum())
        elif "component" in spec:
            flags = c.get(f"{role}_components")
            if flags is None:
                row["availability"] = "eligibility_unavailable"
                row["note"] += (
                    " Component truth is absent; zero padding cannot identify eligibility."
                )
                catalog_rows.append(row)
                continue
            selected = np.asarray(flags)[:, :, TRAJECTORY_COMPONENTS.index(spec["component"])] == 1
            if array.ndim == 3:
                selected = selected[:, None, :]
            eligible = active & selected
            row["unselected_component_count"] = int((active & ~eligible).sum())
        finite = np.isfinite(array)
        row.update(n_eligible=int(eligible.sum()), n_invalid=int((eligible & ~finite).sum()))
        coordinates = np.argwhere(eligible & finite)
        if not len(coordinates):
            row["availability"] = (
                "invalid_only"
                if row["n_invalid"]
                else "drawn_but_unused"
                if row["drawn_but_unused_count"]
                else "no_eligible_component"
                if row["unselected_component_count"]
                else "no_events"
                if spec.get("shock") or array.ndim == 3 and array.shape[1] == 0
                else "no_active_inputs"
                if role != "world" and not row["n_active"]
                else "no_eligible_values"
            )
            catalog_rows.append(row)
            continue
        worlds = coordinates[:, 0]
        unique_worlds, counts = np.unique(worlds, return_counts=True)
        per_world_count = np.zeros(len(cell_ids), dtype=np.int64)
        per_world_count[unique_worlds] = counts
        weights = 1.0 / (len(unique_worlds) * per_world_count[worlds])
        observed = array[tuple(coordinates.T)].astype(np.float64, copy=False)
        n_unique = np.unique(observed).size
        row.update(
            availability="available",
            n_values=len(observed),
            n_worlds=len(unique_worlds),
            n_cells=len(np.unique(cell_ids[unique_worlds])),
            n_unique=n_unique,
            observed_min=float(observed.min()),
            observed_max=float(observed.max()),
            observed_constant=bool(n_unique == 1),
        )
        support_low = np.full(len(observed), spec["low"], dtype=np.float64)
        support_high = np.full(len(observed), spec["high"], dtype=np.float64)
        support_low_error = np.zeros(len(observed))
        support_high_error = np.zeros(len(observed))
        if "conditioning_quantity" in spec:
            quantity = spec["conditioning_quantity"]
            if "prior_cond" in c:
                interval_dtype = np.asarray(c["prior_cond"]).dtype
                intervals = np.asarray(c["prior_cond"], dtype=np.float64)
                lows = intervals[:, PRIOR_COND_LAYOUT.index(f"{quantity}_low")][worlds]
                widths = intervals[:, PRIOR_COND_LAYOUT.index(f"{quantity}_width")][worlds]
                support_low, support_high = lows, lows + widths
                support_low_error = _parameter_bound_tolerance(interval_dtype, lows, lows)
                support_high_error = support_low_error + _parameter_bound_tolerance(
                    interval_dtype, widths, widths
                )
                row["note"] += (
                    f" Conditional low/width labels are {interval_dtype.name}; references use"
                    " rounded stored intervals, not the original sampling intervals."
                )
            else:
                support_low[:] = np.nan
                support_high[:] = np.nan
                row.update(law="conditional_support_unavailable", configured_fixed=False)
                row["note"] += (
                    " Stored per-cell intervals are unavailable; no global-uniform reference is valid."
                )
        support_low_error += _parameter_bound_tolerance(array.dtype, support_low, support_low)
        support_high_error += _parameter_bound_tolerance(array.dtype, support_high, support_high)
        for position, coordinate in enumerate(coordinates):
            if spec.get("shock"):
                event = int(coordinate[1])
                slot = int(c["treatment_shock_index"][coordinate[0], event])
            elif array.ndim == 3:
                event, slot = int(coordinate[1]), int(coordinate[2])
            elif array.ndim == 2:
                event, slot = -1, int(coordinate[1])
            else:
                event, slot = -1, -1
            world_id = int(coordinate[0])
            value_rows.append(
                {
                    "parameter": spec["parameter"],
                    "role": role,
                    "world_id": world_id,
                    "cell_id": int(cell_ids[world_id]),
                    "slot": slot,
                    "event": event,
                    "value": float(observed[position]),
                    "weight": float(weights[position]),
                    "support_low": float(support_low[position]),
                    "support_high": float(support_high[position]),
                    "support_low_tolerance": float(support_low_error[position]),
                    "support_high_tolerance": float(support_high_error[position]),
                }
            )
        catalog_rows.append(row)
    return {
        "catalog": _parameter_table(catalog_rows, "catalog"),
        "values": _parameter_table(value_rows, "values"),
    }


def _parameter_cohorts(reports):
    if not isinstance(reports, Mapping):
        raise TypeError("reports must map cohort labels to parameter_report results")
    if isinstance(reports.get("catalog"), pd.DataFrame) and isinstance(
        reports.get("values"), pd.DataFrame
    ):
        return {"observed": reports}
    corpus = reports.get("corpus")
    if isinstance(corpus, Mapping) and isinstance(corpus.get("cell_id"), np.ndarray):
        return {"observed": reports["parameters"]}
    cohorts = {}
    for label, report in reports.items():
        name = str(label)
        if name in cohorts:
            raise ValueError("cohort labels must remain unique when rendered as strings")
        cohorts[name] = report["parameters"] if "parameters" in report else report
    return cohorts


def _parameter_bound_tolerance(dtype, low, high):
    dtype = np.dtype(dtype)
    if not np.issubdtype(dtype, np.floating):
        return np.zeros(np.broadcast_shapes(np.shape(low), np.shape(high)))
    precision = np.finfo(dtype)
    magnitude = np.maximum(np.abs(low), np.abs(high))
    return 4.0 * precision.eps * magnitude + 4.0 * precision.smallest_subnormal


def _parameter_reference_available(row):
    return (
        row["sampling_unit"] == "accepted_world"
        and row["law"] in ("uniform", "log_uniform", "point_mass", "conditional_uniform")
        and np.isfinite(row["low"])
        and np.isfinite(row["high"])
    )


def _parameter_linear_edges(low, high, bins):
    """Preserve ordinary edges; avoid overflowing a signed float64 span."""
    low, high = float(low), float(high)
    if np.isfinite(high - low):
        edges = np.linspace(low, high, bins + 1)
    else:
        scale = max(abs(low), abs(high))
        edges = np.linspace(low / scale, high / scale, bins + 1) * scale
    edges[0], edges[-1] = low, high
    return np.unique(edges)


def _parameter_tolerated_bounds(low, high, tolerance, high_tolerance=None):
    """Expand bounds only within the representable finite-data domain."""
    limit = float(np.finfo(np.float64).max)
    low, high = np.asarray(low, dtype=float), np.asarray(high, dtype=float)
    high_tolerance = tolerance if high_tolerance is None else high_tolerance
    return (
        low - np.minimum(tolerance, limit + np.minimum(low, 0.0)),
        high + np.minimum(high_tolerance, limit - np.maximum(high, 0.0)),
    )


def _parameter_bin_layout(rows, observed, bins):
    categories = [row["categories"] for row in rows if row["categories"]]
    if categories:
        codes = sorted({float(code) for domain in categories for code in domain.split(",")})
        edges = np.asarray([codes[0] - 0.5, *[value + 0.5 for value in codes]])
        return edges, None, "schema categories; not configured prior coverage"
    known = [row for row in rows if np.isfinite(row["low"]) and np.isfinite(row["high"])]
    unknown_population = any(
        row["n_values"] > 0 and not (np.isfinite(row["low"]) and np.isfinite(row["high"]))
        for row in rows
    )
    if known and not unknown_population:
        low = min(row["low"] for row in known)
        high = max(row["high"] for row in known)
        if low == high:
            return np.empty(0), low, "recorded configured point mass"
        same_support = all(row["low"] == low and row["high"] == high for row in known)
        if same_support and all(row["law"] == "log_uniform" for row in known):
            edges = np.geomspace(low, high, bins + 1)
            basis = "shared recorded log-uniform support; equal-prior-mass bins"
        else:
            edges = _parameter_linear_edges(low, high, bins)
            basis = (
                "shared recorded uniform support; equal-prior-mass bins"
                if same_support and all(row["law"] == "uniform" for row in known)
                else "shared recorded-support domain; references only for recorded nominal laws"
            )
        edges[0], edges[-1] = low, high
        edges = np.unique(edges)
        return edges, None, basis
    if not len(observed):
        return np.empty(0), None, "no finite observed domain; configured support unavailable"
    low, high = float(observed.min()), float(observed.max())
    if known:
        low = min(low, *(row["low"] for row in known))
        high = max(high, *(row["high"] for row in known))
    if low == high:
        width = max(abs(low) * 0.05, 0.5)
        limit = float(np.finfo(np.float64).max)
        return (
            np.asarray([max(-limit, low - width), min(limit, high + width)]),
            None,
            "observed constant; configured support unknown",
        )
    return (
        _parameter_linear_edges(low, high, bins),
        None,
        "shared combined observed domain and recorded supports; unknown prior tails unavailable"
        if known
        else "shared observed domain only; configured support unknown",
    )


def _parameter_prior_masses(law, low, high, edges, point):
    if point is not None:
        return np.asarray([float(low < point), float(low == point), float(low > point)])
    n_bins = len(edges) - 1
    if low == high:
        mass = np.zeros(n_bins + 2)
        index = int(np.searchsorted(edges, low, side="right"))
        if low == edges[-1]:
            index = n_bins
        mass[index] = 1.0
        return mass
    if law == "log_uniform":
        cdf = np.zeros(len(edges))
        positive = edges > 0
        clipped = np.clip(edges[positive], low, high)
        if low / high > 0.5:
            cdf[positive] = np.log1p((clipped - low) / low) / np.log1p((high - low) / low)
        else:
            cdf[positive] = (np.log(clipped) - np.log(low)) / (np.log(high) - np.log(low))
    else:
        clipped = np.clip(edges, low, high)
        low, high = float(low), float(high)
        if np.isfinite(high - low):
            cdf = (clipped - low) / (high - low)
        else:
            scale = max(abs(low), abs(high))
            cdf = (clipped / scale - low / scale) / (high / scale - low / scale)
    return np.r_[cdf[0], np.diff(cdf), 1.0 - cdf[-1]]


def _parameter_expected(row, values, edges, point):
    size = 3 if point is not None else max(len(edges) + 1, 0)
    if not _parameter_reference_available(row) or point is None and not len(edges):
        return np.full(size, np.nan)
    if row["law"] != "conditional_uniform":
        return _parameter_prior_masses(row["law"], row["low"], row["high"], edges, point)
    intervals = values[["world_id", "support_low", "support_high"]].drop_duplicates("world_id")
    if (
        intervals.empty
        or not np.isfinite(intervals[["support_low", "support_high"]].to_numpy()).all()
    ):
        return np.full(size, np.nan)
    masses = np.zeros(size)
    for interval in intervals.itertuples(index=False):
        masses += _parameter_prior_masses(
            "uniform", interval.support_low, interval.support_high, edges, point
        )
    return masses / len(intervals)


def _parameter_observed_bins(row, values, edges, point):
    size = 3 if point is not None else max(len(edges) + 1, 0)
    if point is None and not len(edges):
        return np.empty(0, dtype=np.int64), np.empty(0), np.empty(0, dtype=np.int64)
    if values.empty:
        return np.zeros(size, dtype=np.int64), np.full(size, np.nan), np.empty(0, dtype=np.int64)
    observed = values["value"].to_numpy()
    weights = values["weight"].to_numpy()
    if point is not None:
        tolerance = _parameter_bound_tolerance(row["dtype"], point, point)
        lower, upper = _parameter_tolerated_bounds(point, point, tolerance)
        indexes = np.where(observed < lower, 0, np.where(observed > upper, 2, 1))
    else:
        low, high = edges[0], edges[-1]
        lower, upper = _parameter_tolerated_bounds(
            low,
            high,
            _parameter_bound_tolerance(row["dtype"], low, low),
            _parameter_bound_tolerance(row["dtype"], high, high),
        )
        clipped = observed.copy()
        clipped[(clipped < low) & (clipped >= lower)] = low
        clipped[(clipped > high) & (clipped <= upper)] = high
        indexes = np.searchsorted(edges, clipped, side="right")
        indexes[clipped == high] = len(edges) - 1
    return (
        np.bincount(indexes, minlength=size),
        np.bincount(indexes, weights=weights, minlength=size),
        indexes,
    )


def parameter_bin_report(reports, bins=5):
    """Shared cohort bins and descriptive, equal-world parameter mass summaries.

    reports may be a single parameter report, or a mapping of cohort names to
    parameter reports (also accepting quality_report dictionaries). Outside
    bins keep masses normalized over all finite values. empty_bins refers only
    to core bins and is unavailable when there is no eligible finite population.
    References describe recorded nominal laws, not the iid law of accepted data.
    """
    if isinstance(bins, (bool, np.bool_)) or not isinstance(bins, (int, np.integer)) or bins < 1:
        raise ValueError("bins must be a positive integer")
    cohorts = _parameter_cohorts(reports)
    catalogs = {
        name: report["catalog"].set_index("parameter", drop=False)
        for name, report in cohorts.items()
    }
    parameters = sorted(
        {str(parameter) for catalog in catalogs.values() for parameter in catalog.index}
    )
    bin_rows, summary_rows = [], []
    for parameter in parameters:
        records = [
            catalog.loc[parameter].to_dict()
            for catalog in catalogs.values()
            if parameter in catalog.index
        ]
        observed_parts = [
            report["values"].loc[report["values"]["parameter"] == parameter, "value"].to_numpy()
            for report in cohorts.values()
        ]
        observed = np.concatenate(observed_parts) if observed_parts else np.empty(0)
        edges, point, basis = _parameter_bin_layout(records, observed, int(bins))
        n_core = 1 if point is not None else max(len(edges) - 1, 0)
        if point is not None:
            intervals = [
                (-np.inf, point, "below_point"),
                (point, point, "point_mass"),
                (point, np.inf, "above_point"),
            ]
        elif len(edges):
            intervals = [(-np.inf, edges[0], "below_range")]
            intervals += [(left, right, "in_range") for left, right in zip(edges[:-1], edges[1:])]
            intervals += [(edges[-1], np.inf, "above_range")]
        else:
            intervals = []
        for cohort, report in cohorts.items():
            row = (
                catalogs[cohort].loc[parameter].to_dict()
                if parameter in catalogs[cohort].index
                else records[0]
                | {
                    "availability": "unavailable",
                    "law": "unknown",
                    "low": np.nan,
                    "high": np.nan,
                    "configured_fixed": False,
                    "n_invalid": 0,
                    "drawn_but_unused_count": 0,
                    "unselected_component_count": 0,
                }
            )
            values = report["values"].loc[report["values"]["parameter"] == parameter]
            counts, masses, indexes = _parameter_observed_bins(row, values, edges, point)
            expected = _parameter_expected(row, values, edges, point)
            for index, (left, right, bin_kind) in enumerate(intervals):
                bin_rows.append(
                    {
                        "cohort": cohort,
                        "parameter": parameter,
                        "role": row["role"],
                        "bin": index,
                        "left": left,
                        "right": right,
                        "bin_kind": bin_kind,
                        "bin_basis": basis,
                        "count": int(counts[index]),
                        "mass": float(masses[index]),
                        "expected_mass": float(expected[index]),
                    }
                )
            has_values = not values.empty
            n_unique = values["value"].nunique()
            has_expected = bool(len(expected) and np.isfinite(expected).all())
            below_support = above_support = outside_count = pd.NA
            outside_mass = np.nan
            if has_values and np.isfinite(row["low"]) and np.isfinite(row["high"]):
                if row["law"] == "conditional_uniform":
                    lows, highs = (
                        values["support_low"].to_numpy(),
                        values["support_high"].to_numpy(),
                    )
                    low_error, high_error = (
                        values["support_low_tolerance"].to_numpy(),
                        values["support_high_tolerance"].to_numpy(),
                    )
                else:
                    lows, highs = row["low"], row["high"]
                    low_error = _parameter_bound_tolerance(row["dtype"], lows, lows)
                    high_error = _parameter_bound_tolerance(row["dtype"], highs, highs)
                if np.isfinite(lows).all() and np.isfinite(highs).all():
                    lower, upper = _parameter_tolerated_bounds(lows, highs, low_error, high_error)
                    below = values["value"].to_numpy() < lower
                    above = values["value"].to_numpy() > upper
                    below_support, above_support = int(below.sum()), int(above.sum())
                    outside_count = int((below | above).sum())
                    outside_mass = float(values.loc[below | above, "weight"].sum())
            core_counts = counts[1:-1] if len(counts) else np.empty(0)
            deviation = np.abs(masses - expected) if has_values and has_expected else np.empty(0)
            reference_note = (
                "recorded nominal configured law; accepted draws are filter-conditioned, not an iid fit test"
                if has_expected
                else "cell-level structure is repeated within cells; no independent-world prior overlay"
                if row["sampling_unit"] == "generation_cell"
                else "configuration setting only; no empirical sampled population"
                if row["sampling_unit"] == "configuration"
                else "no justified nominal prior mass reference"
            )
            if row["law"] == "conditional_uniform" and has_expected:
                reference_note = "equal-world nominal mixture from rounded stored low/width labels; original intervals unavailable; acceptance remains filter-conditioned"
            summary_rows.append(
                {
                    "cohort": cohort,
                    "parameter": parameter,
                    "role": row["role"],
                    "label": row["label"],
                    "units": row["units"],
                    "availability": row["availability"],
                    "kind": row["kind"],
                    "law": row["law"],
                    "support_source": row["support_source"],
                    "low": row["low"],
                    "high": row["high"],
                    "sampling_unit": row["sampling_unit"],
                    "bin_basis": basis,
                    "n_values": len(values),
                    "n_worlds": values["world_id"].nunique(),
                    "n_cells": values["cell_id"].nunique(),
                    "n_invalid": row["n_invalid"],
                    "drawn_but_unused_count": row["drawn_but_unused_count"],
                    "unselected_component_count": row["unselected_component_count"],
                    "n_unique": n_unique,
                    "observed_min": float(values["value"].min()) if has_values else np.nan,
                    "observed_max": float(values["value"].max()) if has_values else np.nan,
                    "observed_constant": bool(has_values and n_unique == 1),
                    "configured_fixed": row["configured_fixed"],
                    "n_bins": n_core,
                    "occupied_bins": int((core_counts > 0).sum()) if has_values else pd.NA,
                    "empty_bins": int((core_counts == 0).sum()) if has_values else pd.NA,
                    "occupied_bin_fraction": float((core_counts > 0).mean())
                    if has_values and n_core
                    else np.nan,
                    "max_bin_mass": float(masses.max()) if has_values and len(masses) else np.nan,
                    "max_abs_expected_deviation": float(deviation.max())
                    if len(deviation)
                    else np.nan,
                    "total_variation_from_expected": float(deviation.sum() / 2)
                    if len(deviation)
                    else np.nan,
                    "n_below_support": below_support,
                    "n_above_support": above_support,
                    "n_outside_support": outside_count,
                    "outside_support_mass": outside_mass,
                    "n_below_shared_bins": int(counts[0]) if len(counts) else 0,
                    "n_above_shared_bins": int(counts[-1]) if len(counts) else 0,
                    "reference_note": reference_note,
                    "coverage_limitation": (
                        "No finite applicable population; coverage/concentration are unavailable."
                        if not has_values
                        else "Observed-domain coverage cannot assess unobserved configured support."
                        if not (np.isfinite(row["low"]) and np.isfinite(row["high"]))
                        else "Descriptive coverage only; changing world count does not guarantee uniformity or improvement."
                    ),
                }
            )
    return {
        "bins": _parameter_table(bin_rows, "bins"),
        "summary": _parameter_table(summary_rows, "summary"),
    }


def _parameter_interval_labels(rows):
    labels = []
    for row in rows.itertuples(index=False):
        if row.bin_kind in ("below_range", "below_point"):
            labels.append(f"< {row.right:.4g}")
        elif row.bin_kind in ("above_range", "above_point"):
            labels.append(f"> {row.left:.4g}")
        elif row.bin_kind == "point_mass":
            labels.append(f"= {row.left:.4g}")
        else:
            labels.append(f"{row.left:.4g}–{row.right:.4g}")
    return labels


def plot_parameter_distributions(reports):
    """Render all catalog families, with separate raw-unit and shared-coverage facets.

    A mapping such as {"8 worlds": baseline["parameters"], "100 worlds":
    scaled["parameters"]} produces directly comparable edges. Returns the bin
    report displayed by the parent; plotting never depends on generation state.
    """
    cohorts = _parameter_cohorts(reports)
    binned = parameter_bin_report(cohorts)
    catalogs = {
        name: report["catalog"].set_index("parameter", drop=False)
        for name, report in cohorts.items()
    }
    parameters = []
    for catalog in catalogs.values():
        for parameter in catalog.index:
            if parameter not in parameters:
                parameters.append(parameter)
    observed_parameters = {
        parameter for report in cohorts.values() for parameter in report["values"]["parameter"]
    }
    parameters = [parameter for parameter in parameters if parameter in observed_parameters]
    if not parameters:
        fig, ax = plt.subplots(figsize=(9, 3), layout="constrained")
        ax.text(
            0.5,
            0.5,
            "No finite applicable parameter values\n"
            "Unavailable truth, unused families and absent components remain in the catalog",
            ha="center",
            va="center",
        )
        ax.set_axis_off()
        plt.show()
        plt.close(fig)
        return binned
    groups = {}
    for parameter in parameters:
        row = next(
            catalog.loc[parameter] for catalog in catalogs.values() if parameter in catalog.index
        )
        groups.setdefault(str(row["group"]), []).append(parameter)
    colors = plt.get_cmap("tab10")
    for group, group_parameters in groups.items():
        for page_start in range(0, len(group_parameters), 3):
            page = group_parameters[page_start : page_start + 3]
            fig, axes = plt.subplots(
                len(page), 2, figsize=(14, 4 * len(page)), squeeze=False, layout="constrained"
            )
            fig.suptitle(
                f"{group} · page {page_start // 3 + 1}\n"
                "Equal-world empirical mass; nominal references are not iid acceptance-law tests",
                fontsize=12,
            )
            for position, parameter in enumerate(page):
                raw_ax, coverage_ax = axes[position]
                records = [
                    catalog.loc[parameter].to_dict()
                    for catalog in catalogs.values()
                    if parameter in catalog.index
                ]
                representative = records[0]
                parts = [
                    report["values"]
                    .loc[report["values"]["parameter"] == parameter, "value"]
                    .to_numpy()
                    for report in cohorts.values()
                ]
                joint_values = np.concatenate(parts) if parts else np.empty(0)
                if representative["categories"]:
                    raw_edges, raw_point, _ = _parameter_bin_layout(records, joint_values, 5)
                else:
                    raw_edges, raw_point, _ = _parameter_bin_layout([], joint_values, 5)
                raw_units = representative["units"]
                plot_edges = raw_edges
                magnitude = float(np.max(np.abs(raw_edges)))
                precision = np.finfo(np.float64)
                if magnitude and (
                    magnitude > np.sqrt(precision.max) or magnitude < np.sqrt(precision.tiny)
                ):
                    plot_edges = raw_edges / magnitude
                    raw_units = f"{raw_units} / {magnitude:.4g}"
                parameter_bins = binned["bins"].loc[binned["bins"]["parameter"] == parameter]
                core = parameter_bins["bin_kind"].isin(("in_range", "point_mass"))
                outside_present = parameter_bins.groupby("bin", sort=True)["count"].sum() > 0
                keep = core | parameter_bins["bin"].isin(outside_present.index[outside_present])
                plotted_bins = parameter_bins.loc[keep]
                prototype = plotted_bins.loc[plotted_bins["cohort"] == next(iter(cohorts))]
                x = np.arange(len(prototype))
                width = 0.8 / max(len(cohorts), 1)
                empty_messages = []
                for cohort_index, (cohort, report) in enumerate(cohorts.items()):
                    row = (
                        catalogs[cohort].loc[parameter].to_dict()
                        if parameter in catalogs[cohort].index
                        else representative
                        | {
                            "availability": "unavailable",
                            "law": "unknown",
                            "low": np.nan,
                            "high": np.nan,
                        }
                    )
                    values = report["values"].loc[report["values"]["parameter"] == parameter]
                    summary = (
                        binned["summary"]
                        .loc[
                            (binned["summary"]["parameter"] == parameter)
                            & (binned["summary"]["cohort"] == cohort)
                        ]
                        .iloc[0]
                    )
                    color = colors(cohort_index % 10)
                    label = f"{cohort}: {len(values)} values / {summary['n_worlds']} worlds / {summary['n_cells']} cells"
                    if values.empty:
                        empty_messages.append(f"{cohort}: {row['availability']}")
                        continue
                    counts, raw_mass, _ = _parameter_observed_bins(
                        row, values, raw_edges, raw_point
                    )
                    raw_ax.stairs(
                        raw_mass[1:-1], plot_edges, fill=True, alpha=0.28, color=color, label=label
                    )
                    raw_expected = _parameter_expected(row, values, raw_edges, raw_point)
                    reference_label = f"{cohort}: nominal {row['law']}"
                    if row["law"] == "conditional_uniform":
                        reference_label += " (rounded interval labels)"
                    if len(raw_expected) and np.isfinite(raw_expected).all():
                        raw_ax.stairs(
                            raw_expected[1:-1],
                            plot_edges,
                            color=color,
                            linestyle="--",
                            linewidth=1.8,
                            label=f"{reference_label} mass (not density)",
                        )
                    cohort_bins = plotted_bins.loc[plotted_bins["cohort"] == cohort]
                    offset = (cohort_index - (len(cohorts) - 1) / 2) * width
                    coverage_ax.bar(
                        x + offset,
                        cohort_bins["mass"],
                        width=width,
                        color=color,
                        alpha=0.8,
                        label=label,
                    )
                    expected = cohort_bins["expected_mass"].to_numpy()
                    if len(expected) and np.isfinite(expected).all():
                        coverage_ax.plot(
                            x + offset,
                            expected,
                            "o--",
                            color=color,
                            linewidth=1.5,
                            label=reference_label,
                        )
                title = f"{representative['label']} [{representative['units']}]"
                raw_ax.set(
                    title=title + "\nshared observed-domain raw-value bins",
                    xlabel=raw_units,
                    ylabel="equal-world bin mass",
                )
                coverage_ax.set(
                    title=title + "\nshared coverage bins",
                    ylabel="equal-world bin mass",
                    xticks=x,
                    xticklabels=_parameter_interval_labels(prototype),
                )
                coverage_ax.tick_params(axis="x", rotation=25, labelsize=8)
                for ax in (raw_ax, coverage_ax):
                    ax.set_ylim(0, 1.12)
                    ax.tick_params(labelsize=8)
                    ax.title.set_fontsize(9)
                    if empty_messages:
                        ax.text(
                            0.02,
                            0.98,
                            "\n".join(empty_messages),
                            transform=ax.transAxes,
                            va="top",
                            fontsize=8,
                            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
                        )
                    if ax.get_legend_handles_labels()[0]:
                        ax.legend(fontsize=7, loc="upper right")
                basis = str(
                    binned["summary"]
                    .loc[binned["summary"]["parameter"] == parameter, "bin_basis"]
                    .iloc[0]
                )
                coverage_ax.set_xlabel(basis, fontsize=8)
                if all(
                    bool(row["observed_constant"]) and not bool(row["configured_fixed"])
                    for row in records
                    if row["n_values"]
                ):
                    raw_ax.text(
                        0.02,
                        0.72,
                        "Observed constant only; not a configured point mass",
                        transform=raw_ax.transAxes,
                        fontsize=8,
                    )
                if representative["category_labels"]:
                    coverage_ax.text(
                        0.02,
                        0.62,
                        representative["category_labels"].replace("; ", "\n"),
                        transform=coverage_ax.transAxes,
                        fontsize=7,
                        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
                    )
            plt.show()
            plt.close(fig)
    return binned


def assert_parameter_boundaries():
    """Literal measurement oracles; these diagnostic matrices are not SCM corpora.

    No fixtures are generated from the implementation's masks/aggregation.
    The parent calls this explicitly; defining this cell performs no checks.
    """
    flags = np.zeros((2, 5, len(TRAJECTORY_COMPONENTS)), dtype=np.uint8)
    flags[0, 0, TRAJECTORY_COMPONENTS.index("onset")] = 1
    flags[1, 1, TRAJECTORY_COMPONENTS.index("onset")] = 1
    flags[0, 0, TRAJECTORY_COMPONENTS.index("level_jump")] = 1
    flags[1, :2, TRAJECTORY_COMPONENTS.index("level_jump")] = 1
    flags[0, 1, TRAJECTORY_COMPONENTS.index("trend")] = 1
    flags[1, 1, TRAJECTORY_COMPONENTS.index("trend")] = 1
    flags[:, 0, TRAJECTORY_COMPONENTS.index("seasonal")] = 1
    fixture = {
        "cell_id": np.asarray([9, 9], dtype=np.int32),
        "treatment_active_mask": np.asarray([[1, 1, 0, 0, 0], [1, 1, 1, 1, 0]], dtype=np.uint8),
        "covariate_active_mask": np.asarray([[1, 0], [1, 1]], dtype=np.uint8),
        "carryover_family": np.asarray([[1, 2, 0, 0, 0], [1, 1, 1, 2, 0]], dtype=np.uint8),
        "sat_family": np.asarray([[3, 1, 0, 0, 0], [3, 3, 3, 1, 0]], dtype=np.uint8),
        "mm_kappa_mult": np.asarray([[1, 66, 0, 0, 0], [1, 10, 100, 77, 0]], dtype=np.float64),
        "carryover_alpha": np.asarray(
            [[0, 0.55, 0, 0, 0], [0.25, 0.5, 0.75, 0.66, 0]], dtype=np.float32
        ),
        "treatment_components": flags,
        "trajectory_treatment_onset_start": np.asarray(
            [[0, 0, 0, 0, 0], [0, 3, 0, 0, 0]], dtype=np.int64
        ),
        "trajectory_treatment_trend_change": np.asarray(
            [[0, 0, 0, 0, 0], [0, -0.5, 0, 0, 0]], dtype=np.float64
        ),
        "trajectory_treatment_seasonal_period": np.asarray(
            [[52, 0, 0, 0, 0], [52, 0, 0, 0, 0]], dtype=np.float64
        ),
        "trajectory_treatment_level_jump_factor": np.asarray(
            [[[1, 0, 0, 0, 0], [2, 0, 0, 0, 0]], [[3, 4, 0, 0, 0], [5, 6, 0, 0, 0]]],
            dtype=np.float64,
        ),
        "covariate_reference_input": np.asarray([[2, 0], [2, 2]], dtype=np.float64),
        "covariate_reference_contribution": np.asarray([[0, 0], [-0.3, 0.3]], dtype=np.float64),
        "rho_zy": np.asarray([[0, 0], [-0.15, 0.15]], dtype=np.float64),
        "diagnostics": {
            "mechanism_priors": {
                "saturation_prior_ranges": {"michaelis_menten": {"kappa_mult": [1, 100]}},
                "mm_scale_prior": "log_uniform",
                "covariate_reference_contribution_range": [-0.3, 0.3],
                "covariate_reference_scale": 2,
            }
        },
    }
    covariate_flags = np.zeros((2, 2, len(TRAJECTORY_COMPONENTS)), dtype=np.uint8)
    covariate_flags[0, 0, TRAJECTORY_COMPONENTS.index("level_jump")] = 1
    covariate_flags[1, :, TRAJECTORY_COMPONENTS.index("level_jump")] = 1
    fixture["covariate_components"] = covariate_flags
    fixture["trajectory_covariate_level_jump_size"] = np.asarray(
        [[[0, 0], [-1, 0], [1, 0]], [[2, -2], [3, -3], [4, -4]]], dtype=np.float64
    )
    result = parameter_report(fixture)
    catalog = result["catalog"].set_index("parameter")
    values = result["values"]
    mm = values.loc[values["parameter"] == "mm_kappa_mult"]
    assert list(mm[["world_id", "slot", "event", "value"]].itertuples(index=False, name=None)) == [
        (0, 0, -1, 1.0),
        (1, 0, -1, 1.0),
        (1, 1, -1, 10.0),
        (1, 2, -1, 100.0),
    ]
    np.testing.assert_allclose(mm["weight"], [0.5, 1 / 6, 1 / 6, 1 / 6])
    np.testing.assert_allclose(mm.groupby("world_id")["weight"].sum(), [0.5, 0.5])
    assert catalog.loc[
        "mm_kappa_mult",
        ["n_values", "n_worlds", "n_cells", "drawn_but_unused_count", "padded_count"],
    ].tolist() == [4, 2, 1, 2, 4]
    assert catalog.loc["carryover_alpha", "dtype"] == "float32"
    alpha = values.loc[values["parameter"] == "carryover_alpha", "value"]
    np.testing.assert_array_equal(alpha, [0.0, 0.25, 0.5, 0.75])
    onset = values.loc[values["parameter"] == "trajectory_treatment_onset_start"]
    assert list(onset[["world_id", "slot", "value"]].itertuples(index=False, name=None)) == [
        (0, 0, 0.0),
        (1, 1, 3.0),
    ]
    trend = values.loc[values["parameter"] == "trajectory_treatment_trend_change", "value"]
    np.testing.assert_array_equal(trend, [0.0, -0.5])
    jumps = values.loc[values["parameter"] == "trajectory_treatment_level_jump_factor"]
    assert list(
        jumps[["world_id", "event", "slot", "value"]].itertuples(index=False, name=None)
    ) == [
        (0, 0, 0, 1.0),
        (0, 1, 0, 2.0),
        (1, 0, 0, 3.0),
        (1, 0, 1, 4.0),
        (1, 1, 0, 5.0),
        (1, 1, 1, 6.0),
    ]
    np.testing.assert_allclose(jumps["weight"], [0.25, 0.25, 0.125, 0.125, 0.125, 0.125])
    assert catalog.loc[
        "trajectory_treatment_level_jump_factor", ["n_values", "n_worlds", "n_cells"]
    ].tolist() == [6, 2, 1]
    covariate_jumps = values.loc[values["parameter"] == "trajectory_covariate_level_jump_size"]
    assert list(
        covariate_jumps[["world_id", "event", "slot", "value"]].itertuples(index=False, name=None)
    ) == [
        (0, 0, 0, 0.0),
        (0, 1, 0, -1.0),
        (0, 2, 0, 1.0),
        (1, 0, 0, 2.0),
        (1, 0, 1, -2.0),
        (1, 1, 0, 3.0),
        (1, 1, 1, -3.0),
        (1, 2, 0, 4.0),
        (1, 2, 1, -4.0),
    ]
    np.testing.assert_allclose(
        covariate_jumps["weight"],
        [1 / 6, 1 / 6, 1 / 6, 1 / 12, 1 / 12, 1 / 12, 1 / 12, 1 / 12, 1 / 12],
    )
    assert catalog.loc[
        "trajectory_covariate_level_jump_size", ["n_values", "n_worlds", "n_cells", "padded_count"]
    ].tolist() == [9, 2, 1, 3]
    rho = values.loc[values["parameter"] == "rho_zy", "value"]
    np.testing.assert_array_equal(rho, [0.0, -0.15, 0.15])
    np.testing.assert_allclose(
        catalog.loc["rho_zy", ["low", "high"]].to_numpy(dtype=float), [-0.15, 0.15]
    )

    binned = parameter_bin_report({"literal": result}, bins=2)
    mm_bins = binned["bins"].loc[
        (binned["bins"]["parameter"] == "mm_kappa_mult")
        & (binned["bins"]["bin_kind"] == "in_range")
    ]
    np.testing.assert_allclose(mm_bins["left"], [1, 10])
    np.testing.assert_allclose(mm_bins["right"], [10, 100])
    np.testing.assert_allclose(mm_bins["expected_mass"], [0.5, 0.5])
    np.testing.assert_allclose(mm_bins["mass"], [2 / 3, 1 / 3])
    subnormal = np.nextafter(0.0, 1.0)
    upper = 1e-300
    midpoint = np.sqrt(subnormal) * np.sqrt(upper)
    np.testing.assert_allclose(
        _parameter_prior_masses(
            "log_uniform", subnormal, upper, np.asarray([subnormal, midpoint, upper]), None
        ),
        [0, 0.5, 0.5, 0],
        atol=1e-12,
    )
    limit = float(np.finfo(np.float64).max)
    signed_edges = _parameter_linear_edges(-limit, limit, 2)
    np.testing.assert_array_equal(signed_edges, [-limit, 0, limit])
    np.testing.assert_allclose(
        _parameter_prior_masses("uniform", -limit, limit, signed_edges, None),
        [0, 0.5, 0.5, 0],
    )
    constant_edges, _, _ = _parameter_bin_layout([], np.asarray([limit]), 5)
    assert np.isfinite(constant_edges).all()
    lower, upper_bound = _parameter_tolerated_bounds(
        np.asarray([-limit, limit]), np.asarray([-limit, limit]), limit * 1e-15
    )
    assert np.isfinite(lower).all() and np.isfinite(upper_bound).all()
    assert mm_bins["count"].tolist() == [2, 2]
    summary = binned["summary"].set_index("parameter")
    np.testing.assert_allclose(
        summary.loc[
            "mm_kappa_mult",
            ["max_bin_mass", "max_abs_expected_deviation", "total_variation_from_expected"],
        ].to_numpy(dtype=float),
        [2 / 3, 1 / 6, 1 / 6],
    )
    assert summary.loc["mm_kappa_mult", "empty_bins"] == 0
    assert summary.loc["mm_kappa_mult", "n_outside_support"] == 0
    point_bins = binned["bins"].loc[binned["bins"]["parameter"] == "covariate_reference_input"]
    np.testing.assert_array_equal(point_bins["mass"], [0, 1, 0])
    np.testing.assert_array_equal(point_bins["expected_mass"], [0, 1, 0])
    assert summary.loc["covariate_reference_input", "configured_fixed"]
    assert summary.loc["covariate_reference_input", "observed_constant"]
    unknown = "trajectory_treatment_seasonal_period"
    assert (
        summary.loc[unknown, "observed_constant"] and not summary.loc[unknown, "configured_fixed"]
    )
    assert pd.isna(summary.loc[unknown, "low"])
    assert pd.isna(summary.loc[unknown, "max_abs_expected_deviation"])
    assert binned["bins"].loc[binned["bins"]["parameter"] == unknown, "expected_mass"].isna().all()
    assert catalog.loc["rw_b_std", "availability"] == "unavailable"
    assert catalog.loc["w_dc", "availability"] == "unavailable"
    assert catalog.loc["trajectory_treatment_offset_stop", "availability"] == "unavailable"

    singleton = parameter_report(
        {
            "cell_id": np.asarray([21], dtype=np.int32),
            "treatment_active_mask": np.ones((1, 1), dtype=np.uint8),
            "covariate_active_mask": np.empty((1, 0), dtype=np.uint8),
            "saturation_scale": np.asarray([[7.0]], dtype=np.float32),
            "diagnostics": {},
        }
    )
    singleton_bins = parameter_bin_report({"singleton": singleton}, bins=2)
    single_summary = singleton_bins["summary"].set_index("parameter").loc["saturation_scale"]
    assert (
        single_summary["n_values"] == 1
        and single_summary["n_worlds"] == 1
        and single_summary["n_cells"] == 1
    )
    assert single_summary["observed_constant"] and not single_summary["configured_fixed"]
    assert pd.isna(single_summary["max_abs_expected_deviation"])
    assert (
        singleton_bins["bins"]
        .loc[singleton_bins["bins"]["parameter"] == "saturation_scale", "expected_mass"]
        .isna()
        .all()
    )

    empty = parameter_report(
        {
            "cell_id": np.empty(0, dtype=np.int32),
            "treatment_active_mask": np.empty((0, 5), dtype=np.uint8),
            "covariate_active_mask": np.empty((0, 2), dtype=np.uint8),
            "sat_family": np.empty((0, 5), dtype=np.uint8),
            "mm_kappa_mult": np.empty((0, 5), dtype=np.float64),
            "diagnostics": fixture["diagnostics"],
        }
    )
    assert empty["values"].empty
    assert empty["values"]["world_id"].dtype == np.dtype("int64")
    assert empty["values"]["weight"].dtype == np.dtype("float64")
    paired = parameter_bin_report({"literal": result, "empty": empty}, bins=2)
    paired_mm = paired["bins"].loc[paired["bins"]["parameter"] == "mm_kappa_mult"]
    for cohort in ("literal", "empty"):
        core = paired_mm.loc[
            (paired_mm["cohort"] == cohort) & (paired_mm["bin_kind"] == "in_range")
        ]
        np.testing.assert_allclose(core["left"], [1, 10])
        np.testing.assert_allclose(core["right"], [10, 100])
    assert paired_mm.loc[paired_mm["cohort"] == "empty", "mass"].isna().all()
    empty_summary = (
        paired["summary"].set_index(["cohort", "parameter"]).loc[("empty", "mm_kappa_mult")]
    )
    assert empty_summary["n_values"] == 0
    for column in (
        "max_bin_mass",
        "occupied_bin_fraction",
        "empty_bins",
        "max_abs_expected_deviation",
        "n_outside_support",
    ):
        assert pd.isna(empty_summary[column])
    both_empty = parameter_bin_report({"first": empty, "second": empty}, bins=2)
    assert both_empty["bins"]["mass"].isna().all()
    assert both_empty["summary"]["max_bin_mass"].isna().all()
    no_cohorts = parameter_bin_report({})
    assert no_cohorts["bins"].empty and no_cohorts["summary"].empty
    assert no_cohorts["bins"]["mass"].dtype == np.dtype("float64")

    rounded = parameter_report(
        {
            "cell_id": np.asarray([30, 31], dtype=np.int32),
            "treatment_active_mask": np.ones((2, 1), dtype=np.uint8),
            "covariate_active_mask": np.empty((2, 0), dtype=np.uint8),
            "sat_family": np.full((2, 1), 3, dtype=np.uint8),
            "mm_kappa_mult": np.asarray([[0.1], [0.3]], dtype=np.float32),
            "diagnostics": {
                "mechanism_priors": {
                    "mm_scale_prior": "uniform",
                    "saturation_prior_ranges": {"michaelis_menten": {"kappa_mult": [0.1, 0.3]}},
                }
            },
        }
    )
    rounded_summary = (
        parameter_bin_report(rounded, bins=2)["summary"].set_index("parameter").loc["mm_kappa_mult"]
    )
    assert rounded_summary["n_outside_support"] == 0
    assert (
        rounded_summary["n_below_shared_bins"] == 0 and rounded_summary["n_above_shared_bins"] == 0
    )
    outside_fixture = fixture | {
        "mm_kappa_mult": np.asarray([[1, 66, 0, 0, 0], [1, 10, 101, 77, 0]], dtype=np.float64)
    }
    outside = parameter_bin_report(parameter_report(outside_fixture), bins=2)
    outside_summary = outside["summary"].set_index("parameter").loc["mm_kappa_mult"]
    assert outside_summary["n_outside_support"] == 1 and outside_summary["n_above_shared_bins"] == 1
    np.testing.assert_allclose(outside_summary["outside_support_mass"], 1 / 6)
    np.testing.assert_allclose(
        outside["bins"].loc[outside["bins"]["parameter"] == "mm_kappa_mult", "mass"].sum(), 1
    )

    invalid_fixture = fixture | {
        "mm_kappa_mult": np.asarray([[1, 66, 0, 0, 0], [1, np.nan, 100, 77, 0]], dtype=np.float64)
    }
    invalid = parameter_report(invalid_fixture)
    invalid_catalog = invalid["catalog"].set_index("parameter").loc["mm_kappa_mult"]
    assert invalid_catalog["n_invalid"] == 1 and invalid_catalog["n_values"] == 3
    invalid_mm = invalid["values"].loc[invalid["values"]["parameter"] == "mm_kappa_mult"]
    np.testing.assert_allclose(invalid_mm["weight"], [0.5, 0.25, 0.25])
    no_component = fixture | {"treatment_components": np.zeros_like(flags)}
    no_component_report = parameter_report(no_component)
    no_component_catalog = no_component_report["catalog"].set_index("parameter")
    assert (
        no_component_catalog.loc["trajectory_treatment_onset_start", "availability"]
        == "no_eligible_component"
    )
    assert no_component_catalog.loc["trajectory_treatment_onset_start", "n_values"] == 0
    assert (
        no_component_catalog.loc["trajectory_treatment_onset_start", "unselected_component_count"]
        == 6
    )

    intervals = np.zeros((3, len(PRIOR_COND_LAYOUT)), dtype=np.float32)
    intervals[:, PRIOR_COND_LAYOUT.index("hill_shape_low")] = [1.1, 1.3, 1.1]
    intervals[:, PRIOR_COND_LAYOUT.index("hill_shape_width")] = 0.25
    conditional_view = {
        "cell_id": np.array([19, 23, 19]),
        "treatment_active_mask": np.ones((3, 1), dtype=np.uint8),
        "covariate_active_mask": np.zeros((3, 1), dtype=np.uint8),
        "sat_family": np.ones((3, 1), dtype=np.uint8),
        "hill_slope": np.array([[1.100000001], [1.549999999], [1.09]], dtype=np.float64),
        "prior_cond": intervals,
        "diagnostics": {
            "mechanism_priors": {"saturation_prior_ranges": {"hill": {"slope": [1, 3]}}},
            "prior_cond": {"supports": {"hill_shape": [1, 3]}},
        },
    }
    conditional = parameter_report(conditional_view)
    row = parameter_bin_report(conditional)["summary"].set_index("parameter").loc["hill_slope"]
    assert (row["n_below_support"], row["n_above_support"], row["n_outside_support"]) == (1, 0, 1)
    np.testing.assert_allclose(row["outside_support_mass"], 1 / 3)
    for labels in (("parameters", "primary"), ("catalog", "values")):
        named = parameter_bin_report(dict(zip(labels, (result, conditional))))
        assert set(named["summary"]["cohort"]) == set(labels)
        counts = named["summary"].set_index(["cohort", "parameter"])
        assert counts.loc[(labels[0], "mm_kappa_mult"), "n_values"] == 4
        assert counts.loc[(labels[1], "hill_slope"), "n_values"] == 3

    width_labels = np.zeros((2, len(PRIOR_COND_LAYOUT)), dtype=np.float32)
    width_labels[:, PRIOR_COND_LAYOUT.index("hill_shape_low")] = 0.01
    width_labels[:, PRIOR_COND_LAYOUT.index("hill_shape_width")] = 0.7
    width_view = {
        "cell_id": np.array([31, 37]),
        "treatment_active_mask": np.ones((2, 1), dtype=np.uint8),
        "covariate_active_mask": np.zeros((2, 1), dtype=np.uint8),
        "sat_family": np.ones((2, 1), dtype=np.uint8),
        "hill_slope": np.array([[0.709999999], [0.7101]], dtype=np.float64),
        "prior_cond": width_labels,
        "diagnostics": {
            "mechanism_priors": {"saturation_prior_ranges": {"hill": {"slope": [0.01, 1]}}},
            "prior_cond": {"supports": {"hill_shape": [0.01, 1]}},
        },
    }
    width_row = (
        parameter_bin_report(parameter_report(width_view))["summary"]
        .set_index("parameter")
        .loc["hill_slope"]
    )
    assert (
        width_row["n_below_support"],
        width_row["n_above_support"],
        width_row["n_outside_support"],
    ) == (0, 1, 1)
    np.testing.assert_allclose(width_row["outside_support_mass"], 1 / 2)

    mixture_labels = np.zeros((2, len(PRIOR_COND_LAYOUT)), dtype=np.float32)
    mixture_labels[:, PRIOR_COND_LAYOUT.index("hill_shape_low")] = [1, 2.5]
    mixture_labels[:, PRIOR_COND_LAYOUT.index("hill_shape_width")] = [1.5, 0.5]
    mixture_view = {
        "cell_id": np.array([41, 43]),
        "treatment_active_mask": np.array([[1, 0, 0], [1, 1, 1]], dtype=np.uint8),
        "covariate_active_mask": np.zeros((2, 1), dtype=np.uint8),
        "sat_family": np.ones((2, 3), dtype=np.uint8),
        "hill_slope": np.array([[1.25, 0, 0], [2.6, 2.8, 3]], dtype=np.float64),
        "prior_cond": mixture_labels,
        "diagnostics": conditional_view["diagnostics"],
    }
    mixture_bins = parameter_bin_report(parameter_report(mixture_view), bins=2)["bins"]
    mixture_bins = mixture_bins.loc[
        (mixture_bins["parameter"] == "hill_slope") & (mixture_bins["bin_kind"] == "in_range")
    ]
    np.testing.assert_allclose(mixture_bins["left"], [1, 2])
    np.testing.assert_allclose(mixture_bins["right"], [2, 3])
    np.testing.assert_allclose(mixture_bins["mass"], [1 / 2, 1 / 2])
    np.testing.assert_allclose(mixture_bins["expected_mass"], [1 / 3, 2 / 3])

    unknown_mm_view = {
        "cell_id": np.array([47, 53, 59]),
        "treatment_active_mask": np.ones((3, 1), dtype=np.uint8),
        "covariate_active_mask": np.zeros((3, 1), dtype=np.uint8),
        "sat_family": np.full((3, 1), 3, dtype=np.uint8),
        "mm_kappa_mult": np.array([[200], [600], [1000]], dtype=np.float64),
    }
    unknown_mm_report = parameter_report(unknown_mm_view)
    empty_known_view = fixture | {
        "treatment_active_mask": np.zeros_like(fixture["treatment_active_mask"])
    }
    for known_report in (result, parameter_report(empty_known_view)):
        mixed_bins = parameter_bin_report({"known": known_report, "unknown": unknown_mm_report})
        mm_bins = mixed_bins["bins"].loc[
            (mixed_bins["bins"]["parameter"] == "mm_kappa_mult")
            & (mixed_bins["bins"]["bin_kind"] == "in_range")
        ]
        unknown_bins = mm_bins.loc[mm_bins["cohort"] == "unknown"]
        np.testing.assert_allclose(unknown_bins["left"], [1, 200.8, 400.6, 600.4, 800.2])
        np.testing.assert_allclose(unknown_bins["right"], [200.8, 400.6, 600.4, 800.2, 1000])
        np.testing.assert_allclose(unknown_bins["mass"], [1 / 3, 0, 1 / 3, 0, 1 / 3])
        assert unknown_bins["expected_mass"].isna().all()

In [ ]:
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter


def _zero_period_frame(records, dtypes):
    """Keep the same column order and types when a population is empty."""
    return pd.DataFrame.from_records(records, columns=list(dtypes)).astype(dtypes)


def zero_period_report(c):
    """Measure exact zeros in active raw C/Z inputs over the full reported window.

    World IDs are archive-row IDs; cell IDs are the stored generation-cell IDs.
    Episode endpoints are inclusive and zero-based. Boundary-touching lengths
    are observed window durations, not inferred complete, unobserved durations.
    Both mean and maximum episode length are unavailable when there is no run.
    Worlds with no active inputs retain explicit rows and unavailable rates;
    they never enter eligible-world denominators. No schedule truth is needed.
    """
    series_records = []
    run_records = []
    world_records = []
    cell_ids = np.asarray(c["cell_id"])
    for role, key, prefix in (
        ("treatment", "treatment_raw", "C"),
        ("covariate", "covariates", "Z"),
    ):
        raw = np.asarray(c[key])
        active = np.asarray(c[f"{role}_active_mask"]) == 1
        n_weeks = raw.shape[1]
        for world_id, cell_id in enumerate(cell_ids):
            slots = np.flatnonzero(active[world_id])
            n_affected = 0
            n_all_zero = 0
            n_zero_weeks = 0
            n_zero_runs = 0
            n_boundary_runs = 0
            for slot in slots:
                zero = raw[world_id, :, slot] == 0.0
                changes = np.diff(np.concatenate(([False], zero, [False])).astype(np.int8))
                starts = np.flatnonzero(changes == 1)
                ends = np.flatnonzero(changes == -1) - 1
                lengths = ends - starts + 1
                zero_weeks = int(zero.sum())
                zero_runs = len(starts)
                assert len(ends) == zero_runs
                assert int(lengths.sum()) == zero_weeks
                left_boundary = starts == 0
                right_boundary = ends == n_weeks - 1
                boundary_runs = int(np.count_nonzero(left_boundary | right_boundary))
                some_zeros = zero_weeks > 0
                all_zero = n_weeks > 0 and zero_weeks == n_weeks
                identity = {
                    "world_id": int(world_id),
                    "cell_id": int(cell_id),
                    "role": role,
                    "slot": int(slot),
                    "input": f"{prefix}{slot + 1}",
                }
                series_records.append(
                    {
                        **identity,
                        "n_weeks": n_weeks,
                        "zero_weeks": zero_weeks,
                        "zero_fraction": zero_weeks / n_weeks if n_weeks else np.nan,
                        "zero_runs": zero_runs,
                        "mean_run_length": float(lengths.mean()) if zero_runs else np.nan,
                        "max_run_length": float(lengths.max()) if zero_runs else np.nan,
                        "no_zeros": not some_zeros,
                        "some_zeros": some_zeros,
                        "all_zero": all_zero,
                    }
                )
                for start, end, length, left, right in zip(
                    starts, ends, lengths, left_boundary, right_boundary
                ):
                    run_records.append(
                        {
                            **identity,
                            "start_week": int(start),
                            "end_week": int(end),
                            "length": int(length),
                            "left_boundary": bool(left),
                            "right_boundary": bool(right),
                        }
                    )
                n_affected += int(some_zeros)
                n_all_zero += int(all_zero)
                n_zero_weeks += zero_weeks
                n_zero_runs += zero_runs
                n_boundary_runs += boundary_runs
            n_series = len(slots)
            observed_weeks = n_series * n_weeks
            classification = (
                "no_eligible_inputs"
                if not n_series
                else "none"
                if not n_affected
                else "all"
                if n_affected == n_series
                else "mixed"
            )
            world_records.append(
                {
                    "world_id": int(world_id),
                    "cell_id": int(cell_id),
                    "role": role,
                    "status": (
                        "no_eligible_inputs"
                        if not n_series
                        else "observed_no_episodes"
                        if not n_zero_runs
                        else "observed"
                    ),
                    "n_series": n_series,
                    "n_no_zeros": n_series - n_affected,
                    "n_affected": n_affected,
                    "n_all_zero": n_all_zero,
                    "affected_fraction": n_affected / n_series if n_series else np.nan,
                    "zero_weeks": n_zero_weeks,
                    "n_observed_weeks": observed_weeks,
                    "zero_week_fraction": (
                        n_zero_weeks / observed_weeks if observed_weeks else np.nan
                    ),
                    "n_zero_runs": n_zero_runs,
                    "n_boundary_runs": n_boundary_runs,
                    "any_zero": bool(n_affected) if n_series else pd.NA,
                    "all_affected": n_affected == n_series if n_series else pd.NA,
                    "classification": classification,
                }
            )
    series = _zero_period_frame(
        series_records,
        {
            "world_id": "int64",
            "cell_id": "int64",
            "role": "object",
            "slot": "int64",
            "input": "object",
            "n_weeks": "int64",
            "zero_weeks": "int64",
            "zero_fraction": "float64",
            "zero_runs": "int64",
            "mean_run_length": "float64",
            "max_run_length": "float64",
            "no_zeros": "bool",
            "some_zeros": "bool",
            "all_zero": "bool",
        },
    )
    runs = _zero_period_frame(
        run_records,
        {
            "world_id": "int64",
            "cell_id": "int64",
            "role": "object",
            "slot": "int64",
            "input": "object",
            "start_week": "int64",
            "end_week": "int64",
            "length": "int64",
            "left_boundary": "bool",
            "right_boundary": "bool",
        },
    )
    worlds = _zero_period_frame(
        world_records,
        {
            "world_id": "int64",
            "cell_id": "int64",
            "role": "object",
            "status": "object",
            "n_series": "int64",
            "n_no_zeros": "int64",
            "n_affected": "int64",
            "n_all_zero": "int64",
            "affected_fraction": "float64",
            "zero_weeks": "int64",
            "n_observed_weeks": "int64",
            "zero_week_fraction": "float64",
            "n_zero_runs": "int64",
            "n_boundary_runs": "int64",
            "any_zero": "boolean",
            "all_affected": "boolean",
            "classification": "object",
        },
    )
    summary_records = []
    for role, key in (("treatment", "treatment_raw"), ("covariate", "covariates")):
        inputs = series[series["role"] == role]
        episodes = runs[runs["role"] == role]
        role_worlds = worlds[worlds["role"] == role]
        eligible = role_worlds[role_worlds["n_series"] > 0]
        n_series = len(inputs)
        n_worlds = len(eligible)
        n_no_zeros = int(inputs["no_zeros"].sum())
        n_some_zeros = int(inputs["some_zeros"].sum())
        n_all_zero = int(inputs["all_zero"].sum())
        worlds_none = int((eligible["classification"] == "none").sum())
        worlds_mixed = int((eligible["classification"] == "mixed").sum())
        worlds_all = int((eligible["classification"] == "all").sum())
        worlds_any = worlds_mixed + worlds_all
        n_zero_runs = len(episodes)
        n_boundary_runs = int((episodes["left_boundary"] | episodes["right_boundary"]).sum())
        n_zero_weeks = int(inputs["zero_weeks"].sum())
        observed_weeks = int(inputs["n_weeks"].sum())
        assert n_no_zeros + n_some_zeros == n_series
        assert worlds_none + worlds_mixed + worlds_all == n_worlds
        assert n_zero_runs == int(inputs["zero_runs"].sum())
        assert n_zero_weeks == int(episodes["length"].sum())
        summary_records.append(
            {
                "role": role,
                "status": (
                    "no_eligible_inputs"
                    if not n_series
                    else "observed_no_episodes"
                    if not n_zero_runs
                    else "observed"
                ),
                "n_worlds_total": len(role_worlds),
                "n_worlds_eligible": n_worlds,
                "n_worlds_ineligible": len(role_worlds) - n_worlds,
                "n_cells_total": int(role_worlds["cell_id"].nunique()),
                "n_cells": int(eligible["cell_id"].nunique()),
                "n_series": n_series,
                "n_no_zeros": n_no_zeros,
                "n_some_zeros": n_some_zeros,
                "n_all_zero": n_all_zero,
                "pooled_no_zeros_share": n_no_zeros / n_series if n_series else np.nan,
                "pooled_some_zeros_share": n_some_zeros / n_series if n_series else np.nan,
                "pooled_all_zero_share": n_all_zero / n_series if n_series else np.nan,
                "equal_world_no_zeros_share": (
                    float((1 - eligible["affected_fraction"]).mean()) if n_worlds else np.nan
                ),
                "equal_world_some_zeros_share": (
                    float(eligible["affected_fraction"].mean()) if n_worlds else np.nan
                ),
                "equal_world_all_zero_share": (
                    float((eligible["n_all_zero"] / eligible["n_series"]).mean())
                    if n_worlds
                    else np.nan
                ),
                "worlds_none": worlds_none,
                "worlds_mixed": worlds_mixed,
                "worlds_all": worlds_all,
                "worlds_any_zero": worlds_any,
                "worlds_none_share": worlds_none / n_worlds if n_worlds else np.nan,
                "worlds_mixed_share": worlds_mixed / n_worlds if n_worlds else np.nan,
                "worlds_all_share": worlds_all / n_worlds if n_worlds else np.nan,
                "world_any_zero_share": worlds_any / n_worlds if n_worlds else np.nan,
                "n_weeks": np.asarray(c[key]).shape[1],
                "n_zero_weeks": n_zero_weeks,
                "n_observed_weeks": observed_weeks,
                "pooled_zero_week_fraction": (
                    n_zero_weeks / observed_weeks if observed_weeks else np.nan
                ),
                "equal_world_zero_week_fraction": (
                    float(eligible["zero_week_fraction"].mean()) if n_worlds else np.nan
                ),
                "n_zero_runs": n_zero_runs,
                "n_boundary_runs": n_boundary_runs,
                "boundary_run_share": n_boundary_runs / n_zero_runs if n_zero_runs else np.nan,
                "mean_run_length": float(episodes["length"].mean()) if n_zero_runs else np.nan,
                "max_run_length": float(episodes["length"].max()) if n_zero_runs else np.nan,
                "series_balance_reference": 0.5,
                "reference_meaning": "user diagnostic reference, not a configured prior",
            }
        )
    summary = _zero_period_frame(
        summary_records,
        {
            "role": "object",
            "status": "object",
            "n_worlds_total": "int64",
            "n_worlds_eligible": "int64",
            "n_worlds_ineligible": "int64",
            "n_cells_total": "int64",
            "n_cells": "int64",
            "n_series": "int64",
            "n_no_zeros": "int64",
            "n_some_zeros": "int64",
            "n_all_zero": "int64",
            "pooled_no_zeros_share": "float64",
            "pooled_some_zeros_share": "float64",
            "pooled_all_zero_share": "float64",
            "equal_world_no_zeros_share": "float64",
            "equal_world_some_zeros_share": "float64",
            "equal_world_all_zero_share": "float64",
            "worlds_none": "int64",
            "worlds_mixed": "int64",
            "worlds_all": "int64",
            "worlds_any_zero": "int64",
            "worlds_none_share": "float64",
            "worlds_mixed_share": "float64",
            "worlds_all_share": "float64",
            "world_any_zero_share": "float64",
            "n_weeks": "int64",
            "n_zero_weeks": "int64",
            "n_observed_weeks": "int64",
            "pooled_zero_week_fraction": "float64",
            "equal_world_zero_week_fraction": "float64",
            "n_zero_runs": "int64",
            "n_boundary_runs": "int64",
            "boundary_run_share": "float64",
            "mean_run_length": "float64",
            "max_run_length": "float64",
            "series_balance_reference": "float64",
            "reference_meaning": "object",
        },
    )
    return {"series": series, "runs": runs, "worlds": worlds, "summary": summary}


def plot_zero_period_distributions(reports):
    """Show both roles for a mapping of cohort labels to zero-period reports.

    Pass {"8 worlds": baseline_report["zeros"], "100 worlds": scaled_report["zeros"]}
    for comparison, or a one-cohort mapping for a file-only report. All cohorts
    share bins within each role. Series histograms are pooled over active inputs;
    episode histograms are pooled over actual episodes, including window edges.
    """
    if not reports:
        print("No zero-period report cohorts supplied; no distributions are available.")
        return
    cohorts = list(reports.items())
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    print(
        "Exact raw == 0 over the full reported window; negative controls are not zeros. "
        "These are accepted-world empirical distributions, not an unconditioned iid prior."
    )
    print(
        "The 50% line is the user's binary-series diagnostic reference, not a configured law. "
        "World-any rates have a different denominator and no 50% expectation."
    )
    print(
        "All cohorts use shared bins per role. Episode endpoints are inclusive; lengths "
        "are observed weeks. A dashed length curve is the boundary-touching part of the "
        "total mass, not another independently normalized distribution or a full unseen duration."
    )
    for role, role_label in (("treatment", "C treatments"), ("covariate", "Z signed controls")):
        views = []
        for label, result in cohorts:
            inputs = result["series"][result["series"]["role"] == role]
            episodes = result["runs"][result["runs"]["role"] == role]
            row = result["summary"].loc[result["summary"]["role"] == role].iloc[0]
            views.append((str(label), inputs, episodes, row))
        fraction_edges = np.linspace(0.0, 1.0, 11)
        max_runs = max(
            (int(inputs["zero_runs"].max()) for _, inputs, _, _ in views if len(inputs)), default=0
        )
        count_edges = np.arange(-0.5, max_runs + 1.5, 1.0)
        max_weeks = max(1, max(int(row["n_weeks"]) for _, _, _, row in views))
        length_edges = np.linspace(0.5, max_weeks + 0.5, min(20, max_weeks) + 1)
        fig, axes = plt.subplots(2, 3, figsize=(17, 10), layout="constrained")
        width = 0.8 / len(views)
        for ax, weighting, title in (
            (axes[0, 0], "pooled", "Pooled binary-series balance\nlabels = counts / active series"),
            (
                axes[0, 1],
                "equal_world",
                "Equal-world binary-series balance\nmean within-world shares; labels = weighted shares",
            ),
        ):
            handles = []
            for index, (label, inputs, _, row) in enumerate(views):
                color = colors[index % len(colors)]
                n_series = int(row["n_series"])
                n_worlds = int(row["n_worlds_eligible"])
                handles.append(
                    Patch(
                        facecolor=color,
                        label=f"{label}: {n_series} series / {n_worlds} eligible worlds",
                    )
                )
                if not n_series:
                    ax.text(
                        0.02,
                        0.04 + 0.08 * index,
                        f"{label}: no eligible active series",
                        transform=ax.transAxes,
                        fontsize=8,
                        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
                    )
                    continue
                positions = np.arange(2) + (index - (len(views) - 1) / 2) * width
                shares = [row[f"{weighting}_no_zeros_share"], row[f"{weighting}_some_zeros_share"]]
                bars = ax.bar(positions, shares, width=width, color=color)
                all_zero_share = row[f"{weighting}_all_zero_share"]
                if all_zero_share > 0:
                    ax.bar(
                        positions[1],
                        all_zero_share,
                        width=width,
                        bottom=shares[1] - all_zero_share,
                        facecolor="none",
                        edgecolor="black",
                        linewidth=0.6,
                        hatch="///",
                    )
                for bar, count, share in zip(
                    bars, (row["n_no_zeros"], row["n_some_zeros"]), shares
                ):
                    annotation = (
                        f"{int(count)}/{n_series}" if weighting == "pooled" else f"{share:.1%}"
                    )
                    ax.annotate(
                        annotation,
                        (bar.get_x() + bar.get_width() / 2, share),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha="center",
                        fontsize=8,
                    )
            reference = ax.axhline(
                0.5,
                color="black",
                linestyle=":",
                linewidth=1,
                label="50% diagnostic reference (not a prior)",
            )
            handles.extend(
                [
                    Patch(
                        facecolor="none",
                        edgecolor="black",
                        hatch="///",
                        label="all-zero subset of any-zero",
                    ),
                    reference,
                ]
            )
            ax.set(
                title=title,
                xticks=[0, 1],
                xticklabels=["no zero weeks", "≥1 zero week (includes all-zero)"],
                ylabel="share",
                ylim=(0, 1.3),
                yticks=[0, 0.25, 0.5, 0.75, 1],
            )
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            ax.legend(handles=handles, loc="upper left", fontsize=7)
        ax = axes[0, 2]
        handles = []
        for index, (label, _, _, row) in enumerate(views):
            color = colors[index % len(colors)]
            n_worlds = int(row["n_worlds_eligible"])
            handles.append(
                Patch(
                    facecolor=color,
                    label=f"{label}: {n_worlds} eligible; {int(row['n_worlds_ineligible'])} no-input worlds excluded",
                )
            )
            if not n_worlds:
                ax.text(
                    0.02,
                    0.04 + 0.08 * index,
                    f"{label}: no eligible worlds for this role",
                    transform=ax.transAxes,
                    fontsize=8,
                    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
                )
                continue
            positions = np.arange(4) + (index - (len(views) - 1) / 2) * width
            counts = [
                row["worlds_none"],
                row["worlds_mixed"],
                row["worlds_all"],
                row["worlds_any_zero"],
            ]
            shares = np.asarray(counts, dtype=float) / n_worlds
            bars = ax.bar(positions, shares, width=width, color=color)
            bars[-1].set_hatch("..")
            for bar, count, share in zip(bars, counts, shares):
                ax.annotate(
                    f"{int(count)}/{n_worlds}",
                    (bar.get_x() + bar.get_width() / 2, share),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    fontsize=8,
                )
        ax.set(
            title="World states + world-any subset\n'all' = every input affected, not every week zero",
            xticks=range(4),
            xticklabels=["none", "mixed", "all inputs\naffected", "world any\n(mixed + all)"],
            ylabel="share of eligible worlds",
            ylim=(0, 1.3),
            yticks=[0, 0.25, 0.5, 0.75, 1],
        )
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.legend(handles=handles, loc="upper left", fontsize=7)
        for index, (label, inputs, episodes, row) in enumerate(views):
            color = colors[index % len(colors)]
            if len(inputs):
                fractions = inputs["zero_fraction"].to_numpy(dtype=float)
                finite = np.isfinite(fractions)
                if finite.any():
                    mass = np.histogram(fractions[finite], bins=fraction_edges)[0] / int(
                        finite.sum()
                    )
                    axes[1, 0].stairs(
                        mass,
                        fraction_edges,
                        color=color,
                        linewidth=2,
                        label=f"{label}: {int(finite.sum())}/{len(inputs)} observed series; {int(row['n_all_zero'])} all-zero",
                    )
                else:
                    axes[1, 0].text(
                        0.02,
                        0.04 + 0.08 * index,
                        f"{label}: no observed weeks in active series",
                        transform=axes[1, 0].transAxes,
                        fontsize=8,
                    )
                count_mass = np.histogram(inputs["zero_runs"].to_numpy(), bins=count_edges)[
                    0
                ] / len(inputs)
                axes[1, 1].stairs(
                    count_mass,
                    count_edges,
                    color=color,
                    linewidth=2,
                    label=f"{label}: {len(inputs)} active series; {int(row['n_no_zeros'])} with 0 episodes",
                )
            else:
                for ax in (axes[1, 0], axes[1, 1]):
                    ax.text(
                        0.02,
                        0.04 + 0.08 * index,
                        f"{label}: no eligible active series",
                        transform=ax.transAxes,
                        fontsize=8,
                        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
                    )
            if len(episodes):
                lengths = episodes["length"].to_numpy()
                boundary = (episodes["left_boundary"] | episodes["right_boundary"]).to_numpy()
                mass = np.histogram(lengths, bins=length_edges)[0] / len(episodes)
                boundary_mass = np.histogram(lengths[boundary], bins=length_edges)[0] / len(
                    episodes
                )
                axes[1, 2].stairs(
                    mass,
                    length_edges,
                    color=color,
                    linewidth=2,
                    label=f"{label}: total {len(episodes)} observed episodes",
                )
                axes[1, 2].stairs(
                    boundary_mass,
                    length_edges,
                    color=color,
                    linewidth=1.5,
                    linestyle="--",
                    label=f"{label}: boundary part {int(boundary.sum())}/{len(episodes)} episodes",
                )
            else:
                message = (
                    "no observed zero episodes" if len(inputs) else "no eligible active series"
                )
                axes[1, 2].text(
                    0.02,
                    0.04 + 0.08 * index,
                    f"{label}: {message}",
                    transform=axes[1, 2].transAxes,
                    fontsize=8,
                    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
                )
        axes[1, 0].set(
            title="Zero-week fraction distribution\n0 and 1 retained; shared 0–1 bins",
            xlabel="fraction of reported weeks exactly zero",
            ylabel="mass among observed active series",
            xlim=(0, 1),
        )
        axes[1, 0].xaxis.set_major_formatter(PercentFormatter(1))
        axes[1, 1].set(
            title="Zero-episode count distribution\ncount 0 retained; shared integer bins",
            xlabel="zero episodes per active series",
            ylabel="mass among active series",
            xlim=(count_edges[0], count_edges[-1]),
        )
        axes[1, 1].xaxis.set_major_locator(MaxNLocator(integer=True))
        axes[1, 2].set(
            title="Observed episode-length distribution\nwindow-boundary part dashed; shared week bins",
            xlabel="observed zero-episode length (weeks; ≥1)",
            ylabel="mass among observed episodes",
            xlim=(length_edges[0], length_edges[-1]),
        )
        axes[1, 2].xaxis.set_major_locator(MaxNLocator(integer=True))
        for ax in axes[1]:
            ax.set(ylim=(0, 1.05))
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            if ax.get_legend_handles_labels()[0]:
                ax.legend(loc="upper right", fontsize=7)
        fig.suptitle(
            f"{role_label} · zero periods and series/world balance\n"
            "Hatching within any-zero bars marks the all-zero subset; world-any is mixed + all, not a fourth state.",
            fontsize=12,
        )
        plt.show()
        plt.close(fig)


def assert_zero_period_boundaries():
    """Check literal diagnostic matrices, not pretend valid generated SCM corpora.

    The expected episodes and the 1/1, 0/3, 1/2 aggregation are independent
    literals. This function deliberately does not execute at definition time.
    """
    examples = (
        ("no_zeros", [1, 2, 3, 4, 5], []),
        ("all_zero", [0, 0, 0, 0, 0], [(0, 4, 5, True, True)]),
        (
            "alternating",
            [0, 2, 0, 3, 0],
            [(0, 0, 1, True, False), (2, 2, 1, False, False), (4, 4, 1, False, True)],
        ),
        ("both_boundaries", [0, 0, 2, 0, 0], [(0, 1, 2, True, False), (3, 4, 2, False, True)]),
        ("first_week_only", [2, 0, 0, 0, 0], [(1, 4, 4, False, True)]),
        ("last_week_only", [0, 0, 0, 0, 2], [(0, 3, 4, True, False)]),
        ("interior", [-2, 0, 0, 0, 1], [(1, 3, 3, False, False)]),
        ("negative_nonzero", [-2, -1, -3, -4, -5], []),
        ("signed_zero", [-2, -0.0, 0.0, 2, -1], [(1, 2, 2, False, False)]),
        ("tiny_nonzero", [-1e-30, 1e-30, -3, 2, 1], []),
    )
    values = np.asarray([values for _, values, _ in examples], dtype=float).T[None, :, :]
    values = np.concatenate((values, np.zeros((1, 5, 1))), axis=2)
    active = np.ones((1, len(examples) + 1), dtype=np.uint8)
    active[0, -1] = 0  # The genuinely zero-padded slot is not an eligible series.
    view = {
        "cell_id": np.array([23], dtype=np.int32),
        "treatment_raw": np.abs(values),
        "treatment_active_mask": active,
        "covariates": values,
        "covariate_active_mask": active,
    }
    measured = zero_period_report(view)
    episode_columns = ["start_week", "end_week", "length", "left_boundary", "right_boundary"]
    for role in ("treatment", "covariate"):
        inputs = measured["series"][measured["series"]["role"] == role].set_index("slot")
        assert inputs.index.tolist() == list(range(len(examples)))
        assert inputs["world_id"].tolist() == [0] * len(examples)
        assert inputs["cell_id"].tolist() == [23] * len(examples)
        for slot, (name, raw, expected) in enumerate(examples):
            row = inputs.loc[slot]
            episodes = measured["runs"].loc[
                (measured["runs"]["role"] == role) & (measured["runs"]["slot"] == slot)
            ]
            actual = list(episodes[episode_columns].itertuples(index=False, name=None))
            assert actual == expected, (role, name, actual, expected)
            expected_weeks = sum(episode[2] for episode in expected)
            assert row["n_weeks"] == 5
            assert row["zero_weeks"] == expected_weeks
            assert row["zero_runs"] == len(expected)
            assert row["zero_fraction"] == expected_weeks / 5
            assert bool(row["no_zeros"]) == (len(expected) == 0)
            assert bool(row["some_zeros"]) == (len(expected) > 0)
            assert bool(row["all_zero"]) == (name == "all_zero")
            if expected:
                assert row["mean_run_length"] == expected_weeks / len(expected)
                assert row["max_run_length"] == max(episode[2] for episode in expected)
            else:
                assert pd.isna(row["mean_run_length"])
                assert pd.isna(row["max_run_length"])
            reconstructed = np.zeros(5, dtype=bool)
            for start, end, length, left, right in actual:
                assert length == end - start + 1
                assert left == (start == 0)
                assert right == (end == 4)
                assert start == 0 or raw[start - 1] != 0
                assert end == 4 or raw[end + 1] != 0
                reconstructed[start : end + 1] = True
            np.testing.assert_array_equal(reconstructed, np.asarray(raw) == 0)
        row = measured["summary"].set_index("role").loc[role]
        assert (row["n_no_zeros"], row["n_some_zeros"], row["n_all_zero"]) == (3, 7, 1)
        assert (row["n_zero_runs"], row["n_boundary_runs"], row["n_zero_weeks"]) == (10, 7, 25)
        assert row["boundary_run_share"] == 7 / 10

    # Both roles have exactly these active inputs; repeated cells must not merge worlds.
    slot_week_values = np.array(
        [
            [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]],
            [[1, 2, 3, 4, 5], [2, 3, 4, 5, 6], [3, 4, 5, 6, 7], [0, 0, 0, 0, 0]],
            [[0, 2, 3, 4, 0], [1, 2, 3, 4, 5], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]],
        ],
        dtype=float,
    )
    active = np.array([[1, 0, 0, 0], [1, 1, 1, 0], [1, 1, 0, 0]], dtype=np.uint8)
    aggregate_view = {
        "cell_id": np.array([11, 11, 29], dtype=np.int32),
        "treatment_raw": slot_week_values.transpose(0, 2, 1),
        "treatment_active_mask": active,
        "covariates": -slot_week_values.transpose(0, 2, 1),
        "covariate_active_mask": active,
    }
    aggregate = zero_period_report(aggregate_view)
    for role in ("treatment", "covariate"):
        inputs = aggregate["series"][aggregate["series"]["role"] == role]
        assert list(inputs[["world_id", "slot"]].itertuples(index=False, name=None)) == [
            (0, 0),
            (1, 0),
            (1, 1),
            (1, 2),
            (2, 0),
            (2, 1),
        ]
        worlds = aggregate["worlds"][aggregate["worlds"]["role"] == role]
        assert worlds["n_series"].tolist() == [1, 3, 2]
        assert worlds["n_affected"].tolist() == [1, 0, 1]
        assert worlds["n_all_zero"].tolist() == [1, 0, 0]
        assert worlds["classification"].tolist() == ["all", "none", "mixed"]
        assert worlds["any_zero"].tolist() == [True, False, True]
        assert worlds["all_affected"].tolist() == [True, False, False]
        np.testing.assert_allclose(worlds["affected_fraction"], [1, 0, 0.5], rtol=0, atol=0)
        np.testing.assert_allclose(worlds["zero_week_fraction"], [1, 0, 0.2], rtol=0, atol=0)
        episodes = aggregate["runs"][aggregate["runs"]["role"] == role]
        assert list(episodes[episode_columns].itertuples(index=False, name=None)) == [
            (0, 4, 5, True, True),
            (0, 0, 1, True, False),
            (4, 4, 1, False, True),
        ]
        row = aggregate["summary"].set_index("role").loc[role]
        expected_counts = {
            "n_worlds_total": 3,
            "n_worlds_eligible": 3,
            "n_worlds_ineligible": 0,
            "n_cells": 2,
            "n_cells_total": 2,
            "n_series": 6,
            "n_no_zeros": 4,
            "n_some_zeros": 2,
            "n_all_zero": 1,
            "worlds_none": 1,
            "worlds_mixed": 1,
            "worlds_all": 1,
            "worlds_any_zero": 2,
            "n_zero_runs": 3,
            "n_boundary_runs": 3,
            "n_zero_weeks": 7,
            "n_observed_weeks": 30,
        }
        for column, expected in expected_counts.items():
            assert row[column] == expected, (role, column, row[column], expected)
        expected_shares = {
            "pooled_no_zeros_share": 4 / 6,
            "pooled_some_zeros_share": 2 / 6,
            "pooled_all_zero_share": 1 / 6,
            "equal_world_no_zeros_share": 0.5,
            "equal_world_some_zeros_share": 0.5,
            "equal_world_all_zero_share": 1 / 3,
            "worlds_none_share": 1 / 3,
            "worlds_mixed_share": 1 / 3,
            "worlds_all_share": 1 / 3,
            "world_any_zero_share": 2 / 3,
            "pooled_zero_week_fraction": 7 / 30,
            "equal_world_zero_week_fraction": 0.4,
            "boundary_run_share": 1,
            "mean_run_length": 7 / 3,
            "max_run_length": 5,
        }
        for column, expected in expected_shares.items():
            np.testing.assert_allclose(
                row[column], expected, rtol=0, atol=1e-15, err_msg=f"{role}: {column}"
            )
        assert inputs.loc[inputs["no_zeros"], "mean_run_length"].isna().all()

    mixed_view = {
        "cell_id": np.array([11, 11, 29, 47], dtype=np.int32),
        "treatment_raw": np.concatenate(
            (aggregate_view["treatment_raw"], np.zeros((1, 5, 4))), axis=0
        ),
        "treatment_active_mask": np.concatenate((active, [[0, 0, 0, 0]]), axis=0),
        "covariates": np.concatenate(
            (aggregate_view["covariates"], np.array([[[-1, 0, 0, 0]]] * 5).transpose(1, 0, 2)),
            axis=0,
        ),
        "covariate_active_mask": np.concatenate((active, [[1, 0, 0, 0]]), axis=0),
    }
    mixed = zero_period_report(mixed_view)
    missing_c = mixed["worlds"].set_index(["role", "world_id"]).loc[("treatment", 3)]
    assert missing_c["status"] == "no_eligible_inputs"
    assert pd.isna(missing_c["any_zero"]) and pd.isna(missing_c["all_affected"])
    assert pd.isna(missing_c["affected_fraction"]) and pd.isna(missing_c["zero_week_fraction"])
    c_row, z_row = (
        mixed["summary"].set_index("role").loc[role] for role in ("treatment", "covariate")
    )
    assert (
        c_row["n_worlds_total"],
        c_row["n_worlds_eligible"],
        c_row["n_cells"],
        c_row["n_cells_total"],
    ) == (4, 3, 2, 3)
    assert (z_row["n_worlds_eligible"], z_row["n_cells"], z_row["n_series"]) == (4, 3, 7)
    for row, rates in ((c_row, [2 / 6, 1 / 2, 2 / 3]), (z_row, [2 / 7, 3 / 8, 1 / 2])):
        np.testing.assert_allclose(
            row[
                ["pooled_some_zeros_share", "equal_world_some_zeros_share", "world_any_zero_share"]
            ].to_numpy(dtype=float),
            rates,
        )

    partially_zero = np.array([[[0, 2, 3, 4, 0], [1, 2, 0, 4, 5]]], dtype=float).transpose(0, 2, 1)
    all_affected = zero_period_report(
        {
            "cell_id": np.array([67]),
            "treatment_raw": partially_zero,
            "covariates": -partially_zero,
            "treatment_active_mask": np.ones((1, 2), dtype=np.uint8),
            "covariate_active_mask": np.ones((1, 2), dtype=np.uint8),
        }
    )
    for role in ("treatment", "covariate"):
        world = all_affected["worlds"].set_index("role").loc[role]
        row = all_affected["summary"].set_index("role").loc[role]
        assert (world["n_series"], world["n_affected"], world["n_all_zero"]) == (2, 2, 0)
        assert world["all_affected"] and world["any_zero"]
        assert world["classification"] == "all"
        assert (row["worlds_all"], row["worlds_mixed"], row["world_any_zero_share"]) == (1, 0, 1)

    no_episode_view = {
        "cell_id": np.array([9], dtype=np.int32),
        "treatment_raw": np.ones((1, 5, 1)),
        "treatment_active_mask": np.ones((1, 1), dtype=np.uint8),
        "covariates": -np.ones((1, 5, 1)),
        "covariate_active_mask": np.ones((1, 1), dtype=np.uint8),
    }
    no_episodes = zero_period_report(no_episode_view)
    assert len(no_episodes["series"]) == 2 and no_episodes["runs"].empty
    assert no_episodes["worlds"]["status"].tolist() == ["observed_no_episodes"] * 2
    assert no_episodes["worlds"]["classification"].tolist() == ["none"] * 2
    for _, row in no_episodes["summary"].iterrows():
        assert row["status"] == "observed_no_episodes"
        assert (row["n_series"], row["n_worlds_eligible"], row["n_zero_runs"]) == (1, 1, 0)
        assert row["pooled_no_zeros_share"] == 1 and row["equal_world_no_zeros_share"] == 1
        assert row["pooled_some_zeros_share"] == 0 and row["world_any_zero_share"] == 0
        assert pd.isna(row["mean_run_length"]) and pd.isna(row["boundary_run_share"])

    empty_rates = [
        "pooled_no_zeros_share",
        "pooled_some_zeros_share",
        "pooled_all_zero_share",
        "equal_world_no_zeros_share",
        "equal_world_some_zeros_share",
        "equal_world_all_zero_share",
        "worlds_none_share",
        "worlds_mixed_share",
        "worlds_all_share",
        "world_any_zero_share",
        "pooled_zero_week_fraction",
        "equal_world_zero_week_fraction",
        "boundary_run_share",
        "mean_run_length",
        "max_run_length",
    ]
    for n_worlds in (2, 0):
        empty_view = {
            "cell_id": np.full(n_worlds, 17, dtype=np.int32),
            "treatment_raw": np.zeros((n_worlds, 5, 2)),
            "treatment_active_mask": np.zeros((n_worlds, 2), dtype=np.uint8),
            "covariates": np.zeros((n_worlds, 5, 2)),
            "covariate_active_mask": np.zeros((n_worlds, 2), dtype=np.uint8),
        }
        empty = zero_period_report(empty_view)
        assert empty["series"].empty and empty["runs"].empty
        assert len(empty["worlds"]) == 2 * n_worlds
        assert empty["worlds"]["classification"].tolist() == ["no_eligible_inputs"] * (2 * n_worlds)
        assert empty["worlds"]["any_zero"].isna().all()
        assert empty["worlds"]["all_affected"].isna().all()
        assert empty["summary"]["status"].tolist() == ["no_eligible_inputs"] * 2
        assert empty["summary"]["n_worlds_total"].tolist() == [n_worlds] * 2
        assert empty["summary"]["n_worlds_ineligible"].tolist() == [n_worlds] * 2
        assert empty["summary"]["n_worlds_eligible"].tolist() == [0, 0]
        assert empty["summary"]["n_cells"].tolist() == [0, 0]
        assert empty["summary"]["n_series"].tolist() == [0, 0]
        assert empty["summary"][empty_rates].isna().all().all()
        for table in ("series", "runs", "worlds", "summary"):
            pd.testing.assert_series_equal(empty[table].dtypes, aggregate[table].dtypes)
    for table in ("series", "runs", "worlds", "summary"):
        pd.testing.assert_series_equal(no_episodes[table].dtypes, aggregate[table].dtypes)

In [ ]:
DESCRIPTOR_FEATURES = (
    "treatment_cv_median",
    "treatment_zero_fraction_median",
    "treatment_roughness_median",
    "treatment_acf_lag_1_median",
    "covariate_level_to_variation_median",
    "treatment_pair_diff_abs_pearson_max",
    "outcome_cv",
)
DESCRIPTOR_LAGS = (1, 4, 13, 26, 52)


def component_prevalence(c):
    block = c["diagnostics"].get("trajectory")
    if block is None:
        return pd.DataFrame(
            columns=[
                "role",
                "component",
                "carried_inputs",
                "active_inputs",
                "share",
                "configured_probability",
            ]
        )
    assert tuple(block["components"]) == TRAJECTORY_COMPONENTS
    rows = []
    for role in ("treatment", "covariate"):
        active = c[f"{role}_active_mask"] == 1
        denominator = int(active.sum())
        counts = c[f"{role}_components"][active].sum(axis=0)
        assert block["n_inputs"][role] == denominator
        for component, count in zip(TRAJECTORY_COMPONENTS, counts):
            stored_share = int(count) / denominator if denominator else 0.0
            assert block["prevalence"][role][component] == stored_share
            rows.append(
                {
                    "role": role,
                    "component": component,
                    "carried_inputs": int(count),
                    "active_inputs": denominator,
                    "share": stored_share if denominator else np.nan,
                    "configured_probability": block["inclusion_probs"][role][component],
                }
            )
    return pd.DataFrame(rows)


def mask_coverage(c):
    block = c["diagnostics"].get("active_count_coverage")
    treatments = c["treatment_active_mask"].sum(axis=1).astype(int)
    controls = c["covariate_active_mask"].sum(axis=1).astype(int)
    if block is not None:
        coverage = pg.active_count_coverage(c)
        for key in ("n_treatments_active", "n_covariates_active", "n_worlds", "n_cells"):
            assert coverage[key] == block[key]
    else:
        # Without allocation metadata, only observed bounds are known from a file.
        treatment_axis = list(range(int(treatments.min()), int(treatments.max()) + 1))
        control_axis = list(range(int(controls.min()), int(controls.max()) + 1))
        coverage = {"n_treatments_active": treatment_axis, "n_covariates_active": control_axis}
    world_counts = np.zeros(
        (len(coverage["n_treatments_active"]), len(coverage["n_covariates_active"])), dtype=int
    )
    cell_counts = np.zeros_like(world_counts)
    for i, n_treatments in enumerate(coverage["n_treatments_active"]):
        for j, n_controls in enumerate(coverage["n_covariates_active"]):
            selected = (treatments == n_treatments) & (controls == n_controls)
            world_counts[i, j] = int(selected.sum())
            cell_counts[i, j] = len(np.unique(c["cell_id"][selected]))
    if block is not None:
        np.testing.assert_array_equal(world_counts, coverage["n_worlds"])
        np.testing.assert_array_equal(cell_counts, coverage["n_cells"])
    coverage.update(n_worlds=world_counts.tolist(), n_cells=cell_counts.tolist())
    return coverage


def design_screen(diagnostics, *, vif_threshold):
    rows = []
    for view in ("levels", "differences"):
        result = diagnostics[view].vif["observed"]
        for position, world_id in enumerate(result.world_ids):
            n_predictors = int(result.eligible[position].sum())
            valid_values = result.vif[position, result.valid[position]]
            varying = result.eligible[position] & ~result.constant[position]
            assessed = (
                n_predictors > 0
                and result.n_time_steps >= 3
                and bool(result.valid[position, varying].all())
            )
            maximum = float(valid_values.max()) if valid_values.size else np.nan
            deficient = int(result.rank[position]) < n_predictors
            near_collinear = bool(valid_values.size and (valid_values >= vif_threshold).any())
            rows.append(
                {
                    "world_id": int(world_id),
                    "view": view,
                    "active_predictors": n_predictors,
                    "rank": int(result.rank[position]),
                    "constant_predictors": int(result.constant[position].sum()),
                    "condition": float(result.condition[position]),
                    "max_vif": maximum,
                    "assessed": assessed,
                    "rank_deficient": deficient,
                    "near_collinear": near_collinear,
                    "flagged": deficient or near_collinear,
                }
            )
    return pd.DataFrame(rows)


def quality_report(
    path, *, seed=31, vif_threshold=10.0, max_bad_share=0.25, design_view="differences"
):
    if not np.isfinite(vif_threshold) or vif_threshold < 1:
        raise ValueError("vif_threshold must be finite and >= 1")
    if not 0 <= max_bad_share < 0.5:
        raise ValueError("max_bad_share must be in [0, 0.5) to require a minority")
    if design_view not in ("levels", "differences"):
        raise ValueError("design_view must be levels or differences")
    c = pg.load_corpus(path)
    errors = pg.DataGenerator.validate_corpus(c)
    if errors:
        raise ValueError("Invalid corpus: " + "; ".join(errors))
    rows = pg.world_descriptors(c, source_id="stored_shard", lags=DESCRIPTOR_LAGS)
    diagnostics = pg.data_diagnostics(
        c, scopes=("nodes", "decomposition"), lags=(1, 4, 13, 26, 52), keep_series=False
    )
    design = design_screen(diagnostics, vif_threshold=vif_threshold)
    screen = design[design["view"] == design_view]
    unassessed = int((~screen["assessed"]).sum())
    bad_share = float(screen["flagged"].mean()) if not unassessed else np.nan
    decision = {
        "view": design_view,
        "vif_threshold": float(vif_threshold),
        "max_bad_share": float(max_bad_share),
        "n_worlds": len(screen),
        "n_unassessed": unassessed,
        "rank_deficient_share": float(screen["rank_deficient"].mean()),
        "near_collinear_share": float(screen["near_collinear"].mean()),
        "flagged_share": bad_share,
        "status": "UNASSESSED" if unassessed else "PASS" if bad_share <= max_bad_share else "FAIL",
    }
    trajectories = measure_inputs(c)
    rng = np.random.default_rng(seed)
    random_worlds = np.sort(
        rng.choice(len(c["cell_id"]), size=min(3, len(c["cell_id"])), replace=False)
    )
    candidates = trajectories[trajectories["world_id"].isin(random_worlds)]
    positions = rng.choice(len(candidates), size=min(6, len(candidates)), replace=False)
    random_inputs = candidates.iloc[positions][["world_id", "role", "slot", "input"]].reset_index(
        drop=True
    )
    return {
        "corpus": c,
        "descriptors": rows,
        "summary": pg.summarize_descriptors(rows, features=DESCRIPTOR_FEATURES),
        "diagnostics": diagnostics,
        "prevalence": component_prevalence(c),
        "coverage": mask_coverage(c),
        "design": design,
        "decision": decision,
        "trajectories": trajectories,
        "inventory": trajectory_inventory(trajectories),
        "random_worlds": random_worlds,
        "random_inputs": random_inputs,
        "parameters": parameter_report(c),
        "zeros": zero_period_report(c),
    }


def assert_report_equal(left, right):
    """Compare numerical and status-aware tables, excluding paths and render IDs."""
    for key in ("prevalence", "design", "trajectories", "inventory", "random_inputs"):
        pd.testing.assert_frame_equal(left[key], right[key], check_exact=True)
    for group in ("parameters", "zeros"):
        assert left[group].keys() == right[group].keys()
        for key in left[group]:
            pd.testing.assert_frame_equal(left[group][key], right[group][key], check_exact=True)
    left_bins = parameter_bin_report({"stored": left["parameters"]})
    right_bins = parameter_bin_report({"stored": right["parameters"]})
    for key in left_bins:
        pd.testing.assert_frame_equal(left_bins[key], right_bins[key], check_exact=True)
    pd.testing.assert_frame_equal(
        left["descriptors"].to_frame(with_status=True),
        right["descriptors"].to_frame(with_status=True),
        check_exact=True,
    )
    pd.testing.assert_frame_equal(
        left["summary"].to_frame(), right["summary"].to_frame(), check_exact=True
    )
    np.testing.assert_array_equal(left["random_worlds"], right["random_worlds"])
    assert left["coverage"] == right["coverage"]
    assert left["decision"] == right["decision"]

In [ ]:
# Report from file: set PYMC_GENERATOR_REPORT_PATH for report-only execution.
report_override = os.environ.get("PYMC_GENERATOR_REPORT_PATH")
REPORT_PATH = Path(report_override) if report_override else CORPUS_PATH
report = quality_report(
    REPORT_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
)
repeat = quality_report(
    REPORT_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
)
assert_report_equal(report, repeat)
print("Report reproduced exactly from the stored file at seed", VISUAL_SEED)

In [ ]:
from pymc_generator.signal_diagnostics import check_signal_gate


def show_saved_plot(plot_fn, source, filename, **kwargs):
    path = RUN_DIR / filename
    plot_fn(source, str(path), **kwargs)
    display(Image(filename=str(path)))


def render_quality_overview(result, *, prefix):
    print("Design screen:", result["decision"])

    display(result["design"].round(3))

    display(result["inventory"])

    if result["prevalence"].empty:
        print("Trajectory component truth unavailable in this archive; no prevalence inferred.")
    else:
        display(result["prevalence"].round(3))
        prevalence = (
            result["prevalence"]
            .pivot(index="component", columns="role", values="share")
            .reindex(TRAJECTORY_COMPONENTS)
        )
        ax = prevalence.plot.bar(
            figsize=(10, 3),
            ylim=(0, 1),
            rot=0,
            ylabel="share of active (world, input) pairs",
            title="Carried components — not a classifier of visible shapes",
        )
        plt.show()
        plt.close(ax.figure)

    coverage = result["coverage"]
    allocation = result["corpus"]["diagnostics"].get("active_count_coverage")
    weight_values = allocation.get("weights") if allocation is not None else None
    stored_weights = np.asarray(weight_values) if weight_values is not None else None
    fig, axes = plt.subplots(1, 2, figsize=(11, 6), layout="constrained")
    for ax, name in zip(axes, ("n_cells", "n_worlds")):
        counts = np.asarray(coverage[name])
        excluded = (
            stored_weights == 0 if stored_weights is not None else np.zeros_like(counts, dtype=bool)
        )
        cmap = plt.colormaps["Blues"].copy()
        cmap.set_bad("#eeeeee")
        image = ax.imshow(
            np.ma.masked_where(excluded, counts),
            cmap=cmap,
            vmin=0,
            vmax=max(1, counts.max()),
        )
        for i, j in np.ndindex(counts.shape):
            ax.text(
                j,
                i,
                "×" if excluded[i, j] else str(counts[i, j]),
                ha="center",
                va="center",
                fontsize=8,
            )
        ax.set(
            xticks=range(len(coverage["n_covariates_active"])),
            xticklabels=coverage["n_covariates_active"],
            yticks=range(len(coverage["n_treatments_active"])),
            yticklabels=coverage["n_treatments_active"],
            xlabel="active controls",
            ylabel="active treatments",
            title=name,
        )
        fig.colorbar(image, ax=ax, shrink=0.65)
    plt.show()
    plt.close(fig)
    occupied = int((np.asarray(coverage["n_cells"]) > 0).sum())
    if stored_weights is not None:
        print("× = zero allocation weight, not an accidental coverage gap.")
        print(
            "Positive-weight combinations reached:", occupied, "/", int((stored_weights > 0).sum())
        )
    if allocation is None:
        print(
            "Allocation metadata unavailable; axes cover observed count bounds, not the intended grid."
        )
    else:
        print("Full configured grid reached:", occupied, "/", np.asarray(coverage["n_worlds"]).size)

    display(result["summary"].to_frame().round(3))
    features_to_plot = DESCRIPTOR_FEATURES[:6]
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), layout="constrained")
    for ax, feature in zip(axes.flat, features_to_plot):
        values = result["descriptors"].column(feature)
        finite = np.isfinite(values)
        ax.hist(values[finite], bins=5, color="#306998", edgecolor="white")
        ax.set(title=f"{feature}\nfinite {finite.sum()}/{len(values)}", ylabel="worlds")
        ax.tick_params(labelsize=8)
        ax.title.set_fontsize(8)
    plt.show()
    plt.close(fig)
    print(
        pg.summarize_descriptors(
            result["descriptors"],
            features=("treatment_cv_median",),
            by=("n_treatments_active", "n_covariates_active"),
        ).table("treatment_cv_median")
    )
    print(result["diagnostics"].contributions.table())
    print(pg.outcome_distributions(result["corpus"]).table(of="share"))

    show_saved_plot(viz.plot_vif_diagnostics, result["diagnostics"], f"{prefix}-vif.png")
    show_saved_plot(
        viz.plot_contribution_diagnostics, result["diagnostics"], f"{prefix}-contributions.png"
    )

    signal = result["corpus"]["diagnostics"].get("signal")
    if signal is None:
        print("Stored signal diagnostics unavailable in this archive.")
    else:
        print("Stored signal summary:", signal)
        print("Default signal gate:", check_signal_gate(signal))

### Descriptor distributions and within-world diagnostics

A descriptor row summarizes **one world** over active inputs. The summary retains finite, infinite, undefined, and ineligible counts. Histograms below show finite values only, with excluded-value counts in the title; the table is the authoritative status-aware report.

Read shape, co-movement, contribution size, and rank together. A schema-valid world can still have little useful treatment signal. The signal gate is a separate truth-based diagnostic, not the raw-input VIF screen.


In [ ]:
def render_distribution_reports(reports):
    """Render complete saved-file diagnostics with common comparison bins."""
    parameter_sets = {name: result["parameters"] for name, result in reports.items()}
    zero_sets = {name: result["zeros"] for name, result in reports.items()}
    catalog = pd.concat(
        [result["catalog"].assign(cohort=name) for name, result in parameter_sets.items()],
        ignore_index=True,
    )
    binned = parameter_bin_report(parameter_sets)
    zero_balance = pd.concat(
        [result["summary"].assign(cohort=name) for name, result in zero_sets.items()],
        ignore_index=True,
    )
    for name, result in reports.items():
        c = result["corpus"]
        stored = c["diagnostics"]
        print(
            name,
            ":",
            len(c["cell_id"]),
            "accepted worlds /",
            len(np.unique(c["cell_id"])),
            "generation cells; evaluated candidates:",
            stored.get("n_draws_evaluated", "not recorded"),
            "rejection rate:",
            stored.get("rejection_rate", "not recorded"),
            "failed draw batches:",
            stored.get("n_draw_failures", "not recorded"),
        )
    print(
        "Parameter catalog: unavailable truth and unused families are explicit. "
        "Numeric observations use equal total weight per applicable world, split "
        "over its eligible inputs/events. World draws share structural cell flags."
    )
    catalog_columns = [
        "cohort",
        "parameter",
        "role",
        "kind",
        "law",
        "availability",
        "support_source",
        "low",
        "high",
        "n_values",
        "n_worlds",
        "n_cells",
        "drawn_but_unused_count",
        "dtype",
    ]
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(catalog[catalog_columns])
        print(
            "Bin coverage/concentration: common cohort edges. Matching recorded uniform/log-uniform "
            "supports and laws use equal-prior-mass bins; other comparisons use a shared domain "
            "with cohort-specific nominal masses. Populated unknown supports retain their "
            "combined observed domain, unioned with any recorded supports, not unobserved tails. Constants are not "
            "configured point masses without stored evidence. No iid p-values."
        )
        display(binned["summary"].round(4))
        print("Zero-free versus any-zero series: pooled and equal-world denominators.")
        display(
            zero_balance[
                [
                    "cohort",
                    "role",
                    "n_cells",
                    "n_worlds_eligible",
                    "n_series",
                    "n_no_zeros",
                    "n_some_zeros",
                    "n_all_zero",
                    "pooled_some_zeros_share",
                    "equal_world_some_zeros_share",
                    "world_any_zero_share",
                ]
            ].round(4)
        )
        print("World classes: 'all' means every input has a zero, not every week is zero.")
        display(
            zero_balance[
                [
                    "cohort",
                    "role",
                    "worlds_none",
                    "worlds_mixed",
                    "worlds_all",
                    "worlds_any_zero",
                ]
            ]
        )
        print("Actual maximal zero episodes: observed lengths, including clipped window edges.")
        display(
            zero_balance[
                [
                    "cohort",
                    "role",
                    "n_zero_weeks",
                    "n_zero_runs",
                    "mean_run_length",
                    "max_run_length",
                    "n_boundary_runs",
                    "boundary_run_share",
                ]
            ].round(4)
        )
    plot_parameter_distributions(parameter_sets)
    plot_zero_period_distributions(zero_sets)
    return {
        "catalog": catalog,
        "parameter_bins": binned["bins"],
        "parameter_summary": binned["summary"],
        "zero_balance": zero_balance,
    }

In [ ]:
render_quality_overview(report, prefix="stored-shard")

### Parameter coverage and zero periods — eight-world diagnostics first

These diagnostics use the **stored archive**, not the generation configuration. The original eight worlds are inspected before the later 100-world run. Matching recorded uniform/log-uniform supports and laws use equal-prior-mass bins; other comparisons use shared domains with cohort-specific nominal masses. Derived values and unknown-support constants have no flat-distribution promise. Numerical observations get equal total weight per applicable world; structural flags still have only four independent cell draws here. All histograms describe worlds accepted by the realism/overflow filters.

Zero means **exact raw value == 0**, not a negative control, a padded slot, a missing schedule, or a small nonzero value. Zero episodes are maximal contiguous runs, with inclusive zero-based endpoints and observed lengths. Boundary-touching episodes may continue beyond the reported window. Series, weeks, episodes, and worlds have separate denominators. The 50% line is a diagnostic reference, not an enforced or configured law.


In [ ]:
assert_parameter_boundaries()
assert_zero_period_boundaries()

distribution_reports = {f"Stored shard ({len(report['corpus']['cell_id'])} worlds)": report}
baseline_override = os.environ.get("PYMC_GENERATOR_BASELINE_PATH")
if baseline_override:
    supplied_baseline = quality_report(
        Path(baseline_override),
        seed=VISUAL_SEED,
        vif_threshold=VIF_THRESHOLD,
        max_bad_share=MAX_BAD_SHARE,
    )
    distribution_reports = {
        f"Baseline ({len(supplied_baseline['corpus']['cell_id'])} worlds)": supplied_baseline,
        f"Primary ({len(report['corpus']['cell_id'])} worlds)": report,
    }
distribution_tables = render_distribution_reports(distribution_reports)

### Seeded visual checks on the eight-world baseline

The report's seed selects worlds **and** input series without replacement. Each row below names its original world ID, role, and slot. For treatments, the schedule envelope is a multiplicative `exp(log_level_shift)`; for controls, it is an additive shift in control units. A gate is separate from the envelope. A treatment shock could override an off-week; this shard has no shocks.

`MANUAL_WORLD_IDS` and `MANUAL_INPUTS` are the knobs for curation. They select **existing** datasets, never generate replacements. We translate the explicit ID list to a boolean `worlds=` mask, preserving original IDs and avoiding integer-mask ambiguity for a two-world shard. Diagnostic tables use archive order; the world plots follow your list order. An empty input list still permits world-only inspection.


In [ ]:
def plot_input_selection(c, selection):
    if selection.empty:
        print("No individual inputs selected; world plots remain available.")
        return
    fig, axes = plt.subplots(
        len(selection),
        2,
        figsize=(12, max(3, 2.1 * len(selection))),
        squeeze=False,
        layout="constrained",
    )
    for row, (world_id, role, slot, label) in enumerate(
        selection.itertuples(index=False, name=None)
    ):
        key = "treatment_raw" if role == "treatment" else "covariates"
        x = c[key][world_id, :, slot]
        axes[row, 0].plot(x, linewidth=1, label=f"world {world_id} · {label}")
        axes[row, 0].set_ylabel("input units")
        axes[row, 0].legend(fontsize=8)
        shift_key = "treatment_log_level_shift" if role == "treatment" else "covariate_level_shift"
        if f"{role}_activity" in c and shift_key in c:
            gate = c[f"{role}_activity"][world_id, :, slot]
            axes[row, 0].fill_between(
                np.arange(len(x)),
                0,
                1,
                where=gate == 0,
                color="grey",
                alpha=0.2,
                transform=axes[row, 0].get_xaxis_transform(),
                label="gate off",
            )
            axes[row, 0].legend(fontsize=8)
            shift = c[shift_key][world_id, :, slot]
            envelope = np.exp(shift) if role == "treatment" else shift
            axes[row, 1].plot(
                envelope,
                color="#9c3e74",
                label="level multiplier" if role == "treatment" else "additive level shift",
            )
            axes[row, 1].legend(fontsize=8)
        else:
            axes[row, 1].text(
                0.5,
                0.5,
                "Schedule truth unavailable in this archive",
                ha="center",
                transform=axes[row, 1].transAxes,
            )
    for ax in axes[-1]:
        ax.set_xlabel("reported week")
    plt.show()
    plt.close(fig)


print("Seeded world IDs:", report["random_worlds"].tolist())
display(report["random_inputs"])
plot_input_selection(report["corpus"], report["random_inputs"])
for world_id in report["random_worlds"]:
    plot_world(report["corpus"], int(world_id))

## 5. Grow to 100 worlds — fifty generation cells

The new run keeps the same priors, active-count mix, 104-week window, and generation seed **29**. Only `n_cells` grows from 4 to **50**; two accepted draws per cell give **100 worlds**, not 25 copies of four structures. The four positive-weight count combinations receive 12–13 cells each; the other 84 combinations remain deliberate exclusions.

The eight-world report above has already run. Keep it as a separate frozen baseline. The changed allocation consumes the RNG differently, so the new 100 worlds are **not** a prefix extension or a paired controlled experiment. More data may reduce sampling variability or reveal tails; it does not guarantee flatter distributions, fewer zeros, or a better design screen. No accepted worlds are removed or resampled to improve these plots.


In [ ]:
from dataclasses import replace

baseline_report = quality_report(
    BASELINE_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
)
scaled_cfg = replace(cfg, n_cells=50)
scaled_generated = pg.sample_prior_predictive(scaled_cfg)
SCALED_PATH = RUN_DIR / "mixed-worlds.npz"
pg.save_corpus(scaled_generated, SCALED_PATH)
scaled_corpus = pg.load_corpus(SCALED_PATH)
assert not pg.DataGenerator.validate_corpus(scaled_corpus)
assert scaled_corpus["treatment_raw"].shape == (100, 104, 15)
assert scaled_corpus["covariates"].shape == (100, 104, 10)
assert len(np.unique(scaled_corpus["cell_id"])) == 50
assert (scaled_corpus["treatment_raw"] >= 0).all()
assert (scaled_corpus["covariates"] < 0).any()
assert not scaled_corpus["latent_unobserved_contribution"].any()
assert not scaled_corpus["indirect_effects_by_source"].any()
for world_id in range(100):
    graph = layout.unpack(scaled_corpus["g"][world_id])
    assert {kind: int(block.sum()) for kind, block in graph.items()} == {
        kind: budget[0] for kind, budget in SPARSE_EDGES.items()
    }
for key, value in scaled_generated.items():
    if isinstance(value, np.ndarray):
        np.testing.assert_array_equal(scaled_corpus[key], value, err_msg=key)
    elif key == "identifiability":
        for name, array in value.items():
            np.testing.assert_array_equal(scaled_corpus[key][name], array, err_msg=name)
assert scaled_corpus["diagnostics"] == {
    key: value for key, value in scaled_generated["diagnostics"].items() if key != "timing"
}
print("100-world primary shard:", SCALED_PATH.name, "| seed:", scaled_cfg.seed)
print("Retained eight-world baseline:", BASELINE_PATH.name)
print(
    "Sampling diagnostics:",
    {
        key: scaled_corpus["diagnostics"][key]
        for key in ("n_draws_evaluated", "rejection_rate", "n_draw_failures")
    },
)

In [ ]:
scaled_report = quality_report(
    SCALED_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
)
assert_report_equal(
    scaled_report,
    quality_report(
        SCALED_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
    ),
)
render_quality_overview(scaled_report, prefix="mixed100")

In [ ]:
distribution_reports = {"8 worlds": baseline_report, "100 worlds": scaled_report}
distribution_tables = render_distribution_reports(distribution_reports)
print(
    "A larger cohort is not automatically more balanced: read shared-bin masses and qualified references."
)

In [ ]:
# Later curation, diversity and the design decision intentionally use the primary 100 worlds.
report = scaled_report
REPORT_PATH = SCALED_PATH
CORPUS_PATH = SCALED_PATH
measurements = report["trajectories"]
print("Seeded 100-world sample IDs:", report["random_worlds"].tolist())
display(report["random_inputs"])
plot_input_selection(report["corpus"], report["random_inputs"])
for world_id in report["random_worlds"]:
    plot_world(report["corpus"], int(world_id))

### Manual selection from the 100-world primary shard

The explicit world/input picks below use the full 100-world archive, independently of the seeded examples. All IDs are original archive positions.


In [ ]:
MANUAL_WORLD_IDS = [0, 99]  # edit this list; original IDs in the 100-world REPORT_PATH
MANUAL_INPUTS = [(0, "treatment", 0), (99, "covariate", 0)]  # zero-based slots, C1 / Z1

manual_ids = np.asarray(MANUAL_WORLD_IDS)
n_available = len(report["corpus"]["cell_id"])
if manual_ids.ndim != 1 or manual_ids.dtype.kind not in "iu" or not manual_ids.size:
    raise ValueError("Select a nonempty list of integer world IDs")
if (
    (manual_ids < 0).any()
    or (manual_ids >= n_available).any()
    or len(np.unique(manual_ids)) != len(manual_ids)
):
    raise ValueError("World IDs must be unique positions within the stored shard")
manual_mask = np.isin(np.arange(n_available), manual_ids)
manual_descriptors = pg.world_descriptors(
    report["corpus"], source_id="stored_shard", worlds=manual_mask, lags=DESCRIPTOR_LAGS
)
manual_diagnostics = pg.data_diagnostics(
    report["corpus"], worlds=manual_mask, scopes=("nodes", "decomposition"), lags=(1, 4)
)
print(manual_descriptors.table(DESCRIPTOR_FEATURES))
print(manual_diagnostics.contributions.table())
manual_rows = []
for world_id, role, slot in MANUAL_INPUTS:
    if world_id not in MANUAL_WORLD_IDS:
        raise ValueError("Manual inputs must belong to the selected worlds")
    if role not in ("treatment", "covariate"):
        raise ValueError("Input role must be treatment or covariate")
    mask = report["corpus"][f"{role}_active_mask"][world_id]
    if not 0 <= slot < len(mask) or not mask[slot]:
        raise ValueError("Choose an active slot, not zero padding")
    manual_rows.append((world_id, role, slot, f"{'C' if role == 'treatment' else 'Z'}{slot + 1}"))
manual_selection = pd.DataFrame(manual_rows, columns=["world_id", "role", "slot", "input"])
plot_input_selection(report["corpus"], manual_selection)
for world_id in MANUAL_WORLD_IDS:
    plot_world(report["corpus"], world_id)

## 6. Generate the trajectory atlas: eight recipes

One small `SCM` per archetype, with two treatments, one signed control, direct effects only, and a disconnected latent. Unlike a training shard, a single world has no cell-level train/validation split; we export it as a **CSV/text/plot audit bundle**, not a fictitious one-world corpus.

The measurement adapter below is just a view of genuine `SCM.data` and `SCM.params["trajectory"]`, **not** a corpus that can be passed to `save_corpus`. It preserves the same raw-input measurement rules. The texture-only `always_on_spikes` world has no schedule block; we leave its component truth unavailable rather than inventing flags.

Presets configure schedules, not deterministic smooth curves. Noise remains visible. A pure treatment ramp changes its **log level**; control trends are additive and can have either sign. The doubling recipe fixes the treatment jump factor to exactly two and uses quiet, parentless controls whose additive jump equals their pinned level.


In [ ]:
ATLAS_EDGES = dict.fromkeys(SPARSE_EDGES, (0, 0))
ATLAS_EDGES.update(cy=(2, 2), zy=(1, 1))
atlas = {}
atlas_paths = {}
for index, name in enumerate(TRAJECTORY_ARCHETYPES):
    prior = pg.make_scm_prior(
        n_treatments=2,
        n_covariates=1,
        n_latent=1,
        n_time_steps=104,
        trajectories=name,
        edge_budget=ATLAS_EDGES,
        **MECHANISMS,
    )
    world = pg.sample_scm(prior, seed=100 + index, name=name)
    assert world is not None
    assert world.identity_error() < 1e-9
    atlas[name] = world
    atlas_paths[name] = pg.write_scm_bundle(world, str(RUN_DIR / name))
print("Atlas bundles:", list(atlas))
print("Each folder has model-facing dataset.csv and separate true_components.csv.")


def scm_measurement_view(world):
    view = {
        "treatment_raw": world.data["treatments"][None],
        "covariates": world.data["covariates"][None],
        "treatment_active_mask": np.ones((1, world.n_treatments), dtype=np.uint8),
        "covariate_active_mask": np.ones((1, world.n_covariates), dtype=np.uint8),
    }
    truth = world.params.get("trajectory")
    if truth is not None:
        for role in ("treatment", "covariate"):
            spec = truth[role]
            view[f"{role}_components"] = np.stack(
                [spec["use"][component] for component in TRAJECTORY_COMPONENTS], axis=-1
            )[None]
            for component, leaves in spec.items():
                if component != "use":
                    for leaf, values in leaves.items():
                        if leaf != "count":
                            view[f"trajectory_{role}_{component}_{leaf}"] = np.asarray(values)[None]
    return view


atlas_measurements = pd.concat(
    [
        measure_inputs(scm_measurement_view(world)).assign(archetype=name)
        for name, world in atlas.items()
    ],
    ignore_index=True,
)
display(
    atlas_measurements[
        [
            "archetype",
            "role",
            "input",
            "zero_fraction",
            "first_nonzero_week",
            "last_nonzero_week",
            "on_runs",
            "magnitude_slope",
            "schedule_trend_change",
            "seasonal_period_weeks",
            "exact_doubling_schedule",
        ]
    ].round(3)
)
for name, world in atlas.items():
    fig, axes = plt.subplots(1, 3, figsize=(12, 2.5), layout="constrained")
    for slot in range(world.n_treatments):
        axes[0].plot(world.data["treatments"][:, slot], label=f"C{slot + 1}", linewidth=1)
    axes[0].set(title=f"{name}: treatments", ylabel="input units")
    axes[0].legend(fontsize=7)
    axes[1].plot(world.data["covariates"][:, 0], color="#9c3e74", linewidth=1, label="Z1")
    axes[1].set(title="signed control", ylabel="input units")
    if "treatment_activity" in world.data:
        axes[2].plot(
            np.exp(world.data["treatment_log_level_shift"][:, 0]),
            label="C1 level multiplier",
            color="#b77c24",
        )
        axes[2].plot(
            world.data["treatment_activity"][:, 0], label="C1 gate", color="grey", alpha=0.7
        )
    else:
        axes[2].text(
            0.5,
            0.5,
            "Texture-only world:\nno stored schedule block",
            transform=axes[2].transAxes,
            ha="center",
        )
    axes[2].set(title="schedule truth (not raw input)")
    if "treatment_activity" in world.data:
        axes[2].legend(fontsize=7)
    for ax in axes:
        ax.set_xlabel("reported week")
    plt.show()
    plt.close(fig)

### Fill a measured gap deliberately

An absent row means **not observed in this finite shard**, not impossible under the library. For example, a continuously drawn jump factor will not reliably hit *exactly* two. The atlas's `level_doubling` recipe pins it to two. Likewise, `ramp_up` combines onset with positive trend; `decay_to_zero` combines negative trend with an exact offset gate. Exponential decay alone approaches zero; the gate supplies exact zeros.


In [ ]:
doubling = atlas["level_doubling"].params["trajectory"]["treatment"]["level_jump"]["factor"]
np.testing.assert_array_equal(doubling, 2.0)
assert atlas_measurements.loc[
    atlas_measurements["archetype"] == "level_doubling", "exact_doubling_schedule"
].any()
mixed_doublings = int(
    measurements.loc[measurements["role"] == "treatment", "exact_doubling_schedule"].sum()
)
print("100-world primary: exact x2 treatment schedules =", mixed_doublings)
print("Atlas: fixed x2 schedule demonstrated.")
for name in ("periodic_on_off", "delayed_start", "ramp_up", "decay_to_zero"):
    world = atlas[name]
    for role, key in (("treatment", "treatments"), ("covariate", "covariates")):
        assert not world.data[key][world.data[f"{role}_activity"] == 0].any()
print("All gated atlas inputs are exact zero in their off-weeks.")

## 7. Diversity between datasets: bins, neighbors, and comparisons

The APIs work on a corpus or a list of `SCM` objects. We keep source IDs distinct so an atlas world and a shard world cannot accidentally share an identity. Known generation-cell IDs let nearest-neighbor search exclude sibling draws.

Distances use a scale fitted on the **mixed reference**, not on the query. The chosen features are observable and nonconstant in this reference; adding a constant or nonfinite feature requires an explicit decision, not silently dropping it. Similar descriptors do not establish duplicate datasets, causal equivalence, or equal usefulness for training.


In [ ]:
atlas_rows = pg.WorldDescriptors.concat(
    [
        pg.world_descriptors([world], source_id=name, lags=DESCRIPTOR_LAGS)
        for name, world in atlas.items()
    ]
)
all_rows = pg.WorldDescriptors.concat([report["descriptors"], atlas_rows])
print(
    pg.summarize_descriptors(
        all_rows,
        features=("treatment_cv_median", "treatment_zero_fraction_median"),
        by=("source_id",),
    ).table("treatment_zero_fraction_median")
)
print(
    pg.bin_counts(
        report["descriptors"],
        {
            "treatment_cv_median": [0, 0.25, 0.5, 1, 2, 5],
            "treatment_zero_fraction_median": [0, 0.1, 0.3, 0.6, 1],
        },
    ).table()
)
SIMILARITY_FEATURES = (
    "treatment_cv_median",
    "treatment_roughness_median",
    "treatment_acf_lag_1_median",
)
within = pg.nearest_worlds(
    report["descriptors"], features=SIMILARITY_FEATURES, exclude_same_group=True
)
print("Nearest world from a different generation cell:")
print(within.table())
print("Atlas worlds relative to the mixed reference, on the same scale:")
print(
    pg.nearest_worlds(
        report["descriptors"], atlas_rows, features=SIMILARITY_FEATURES, scale=within.scale
    ).table()
)
print(
    pg.compare_descriptors(report["descriptors"], atlas_rows, features=SIMILARITY_FEATURES).table(
        "treatment_cv_median"
    )
)

## 8. Mechanism priors, exact replay, template parity, and an oracle

The ninth small world puts **all eight components on every input**. This is a capability check, not a prevalence sample. We keep the same geometric + MM response and parameter-only reference priors.

Reference targets are **nominal**, before edge gates. For MM at reference multiplier `m`, the response is `m / (m + kappa_mult)` and `beta × response = target`. A control uses `rho_zy × reference_input = target`. Neither promises the mean contribution over a noisy realized trajectory. In these unfloored worlds a control's actual effect remains the literal `(g_zy × rho_zy) × Z` on reported controls; saturation affects treatments, not controls.

Legacy corpus features are stored in float32; exact mechanism and continuous schedule leaves are float64. A single `SCM` retains its float64 forward values. We therefore use storage-scale tolerances for corpus reconstruction, **bitwise replay** in the same numerical environment, and tight numerical tolerances for separately compiled model paths.


In [ ]:
all_components = {
    f"{role}_{component}_inclusion_prob": 1.0
    for role in ("treatment", "covariate")
    for component in TRAJECTORY_COMPONENTS
}
replay_edges = dict.fromkeys(SPARSE_EDGES, (0, 0))
replay_edges.update(cy=(3, 3), zy=(2, 2))
replay_cfg = pg.make_scm_prior(
    n_treatments=3,
    n_covariates=2,
    n_latent=1,
    n_time_steps=104,
    trajectories="composable",
    edge_budget=replay_edges,
    **MECHANISMS,
    **all_components,
)
rich = pg.sample_scm(replay_cfg, seed=27, name="all_components")
assert rich is not None
for role in ("treatment", "covariate"):
    assert all(np.asarray(flag).all() for flag in rich.params["trajectory"][role]["use"].values())
replayed = rich.replay()  # executes the generative graph at the recorded accepted primitives
assert set(replayed) == set(rich.data)
for name, expected in rich.data.items():
    np.testing.assert_array_equal(replayed[name], expected, err_msg=name)
print(
    "Exact replay:",
    len(replayed),
    "forward outputs from",
    len(rich.primitive_parameters),
    "recorded primitive draws",
)

p = rich.params
multiplier = rich.cfg.treatment_reference_multiplier
np.testing.assert_allclose(
    p["beta"] * multiplier / (multiplier + p["mm_kappa_mult"]),
    p["treatment_reference_contribution"],
    rtol=3e-14,
    atol=0.0,
)
np.testing.assert_allclose(
    p["rho_zy"] * rich.cfg.covariate_reference_scale,
    p["covariate_reference_contribution"],
    rtol=3e-14,
    atol=0.0,
)
expected_control = (rich.g["g_zy"] * p["rho_zy"])[None, :] * rich.data["covariates"]
np.testing.assert_array_equal(expected_control, rich.data["covariate_contribution"])
print(
    "Reference targets reproduced; literal control contribution verified on float64 reported inputs."
)

active = scaled_corpus["treatment_active_mask"] == 1
np.testing.assert_allclose(
    scaled_corpus["beta"][active] * scaled_corpus["treatment_reference_response"][active],
    scaled_corpus["treatment_reference_contribution"][active],
    rtol=3e-14,
    atol=0.0,
)
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
relative_input = np.linspace(0, 5, 200)
first_graph = layout.unpack(scaled_corpus["g"][0])
for slot in np.flatnonzero(first_graph["cy"]):
    response = (
        scaled_corpus["beta"][0, slot]
        * relative_input
        / (relative_input + scaled_corpus["mm_kappa_mult"][0, slot])
    )
    ax.plot(relative_input, response, label=f"C{slot + 1}")
ax.axvline(
    scaled_cfg.treatment_reference_multiplier,
    color="grey",
    linestyle="--",
    label="reference input",
)
ax.set(
    xlabel="post-carryover input / parameter-only anchor",
    ylabel="nominal contribution",
    title="Near-linear to saturated responses in primary world 0",
)
ax.legend(fontsize=7)
plt.show()
plt.close(fig)
rich_bundle = pg.write_scm_bundle(rich, str(RUN_DIR / "all-components"))
display(Image(filename=str(Path(rich_bundle) / "dag.png")))
display(Image(filename=str(Path(rich_bundle) / "decomposition.png")))
display(Image(filename=str(Path(rich_bundle) / "treatments.png")))

### Reusable template: evaluate the same world, not a second seed

The experimental template compiles at maximum sizes and receives graphs, activity masks, discrete families, and component flags as runtime data. It is **not** a switch on `sample_prior_predictive`, nor a complete accepted-corpus pipeline. Shocks, prior conditioning, and non-fixed confounding-strength ranges are outside its current scope.

Here it recomputes **every forward output of the already accepted world** using that world's exact primitive draws. No extra dataset is sampled. Matching a seed across the ordinary and padded template paths does not align their random variables; same-input evaluation is the relevant comparison.


In [ ]:
from pytensor.graph.replace import clone_replace

from pymc_generator.world_model_template import build_cell_inputs, build_world_model_template


def evaluate_at_primitives(model, names, primitives):
    replacements = {rv: rv.type(name=f"given_{rv.name}") for rv in model.free_RVs}
    expressions = clone_replace(
        [model[name] for name in names], replace=replacements, rebuild_strict=False
    )
    evaluate = pytensor.function(
        list(replacements.values()), expressions, on_unused_input="ignore", mode="FAST_COMPILE"
    )
    values = [np.asarray(primitives[rv.name], dtype=rv.dtype) for rv in replacements]
    return dict(zip(names, evaluate(*values)))


active_inputs = {
    "active_treatment": np.ones(rich.n_treatments),
    "active_covariate": np.ones(rich.n_covariates),
    "active_latent": np.ones(rich.n_latent),
}
cell = build_cell_inputs(rich.cfg, rich.g, active_inputs, rich.extras["structural"])
template, template_names, _ = build_world_model_template(rich.cfg, cell, rich.n_time_steps)
assert set(template_names) == set(rich.data)
template_values = evaluate_at_primitives(template, template_names, rich.primitive_parameters)
for name, expected in rich.data.items():
    if np.issubdtype(expected.dtype, np.integer):
        np.testing.assert_array_equal(template_values[name], expected, err_msg=name)
    else:
        np.testing.assert_allclose(
            template_values[name], expected, rtol=2e-11, atol=2e-11, err_msg=name
        )
print(
    "Template parity:",
    len(template_values),
    "forward outputs evaluated at the same accepted primitives",
)

### Fixed-input oracle: forward truth, without fitting a posterior

An oracle conditions on the **already scheduled observed inputs**; it must not apply the gates or envelopes a second time. We use `latent="sampled"` and bind the same primitive baseline/latent innovations, then compare its contribution and conditional outcome mean with generating truth.

With positive-decay geometric carryover and pre-window burn-in, the first `l_max - 1` reported rows need unpersisted treatment history. The oracle's reproducible likelihood window discards those rows. We compare only that window. Its conditional mean excludes observation noise. The default `latent="marginal"` instead integrates latent walks and cannot be compared with a realized baseline as though they were the same mean.

This demonstrates forward correctness, **not posterior recovery or convergence**. The separate [MMM recovery case study](https://pymc-labs.github.io/pymc-generator/examples/simple-model/) performs posterior fitting.


In [ ]:
oracle = rich.oracle_model(latent="sampled")
oracle_values = evaluate_at_primitives(
    oracle, ("contributions", "outcome_mu"), rich.primitive_parameters
)
warmup = (
    rich.cfg.l_max - 1 if rich.cfg.carryover_burn_in > 0 else 0
)  # this recipe uses positive-decay geometric kernels
np.testing.assert_allclose(
    oracle_values["contributions"][warmup:],
    rich.data["contributions_observed"][warmup:],
    rtol=2e-12,
    atol=2e-12,
)
np.testing.assert_allclose(
    oracle_values["outcome_mu"][warmup:],
    (rich.data["outcome"] - rich.data["outcome_noise"])[warmup:],
    rtol=2e-12,
    atol=2e-12,
)
print("Oracle forward values match on weeks", warmup, "through", rich.n_time_steps - 1)
print("Oracle free variables:", sorted(rv.name for rv in oracle.free_RVs))
print("No MCMC or additional dataset generation was used.")

## 9. Keep the datasets and decide what to change

`mixed-worlds.npz` is the authoritative **100-world primary**; `baseline-8-worlds.npz` retains the original eight worlds. We export one model-facing CSV per world from both, omitting only padded inputs, plus the eight atlas and one replay audit bundles. `manifest.csv` maps **117 source-qualified datasets** to unique model-data paths and their separate truth sources.

A practical loop:

1. Check schema and additive closure.
2. Read active-count coverage and **zero-weight exclusions**.
3. Check parameter coverage against justified priors, zero-episode counts/lengths and series/world balance, carried components, and measured visible shapes.
4. Inspect rank/VIF in differences **and** levels, signal strength, and outcome shares.
5. Plot seeded examples; hand-pick unusual worlds by original ID.
6. Change the prior deliberately, generate another shard, and use `compare_descriptors` on a frozen reference.

Do not equate a missing trajectory in eight worlds with absent generator support. Increase **cells**, not merely draws per cell, to diversify graph sizes and component combinations. Turning features on can change other same-seed draws through graph/RNG layout; identical seeds across recipes are not controlled experiments. With features off, the library's existing golden checks guard its original laws and RNG streams. Inside one compiled template, runtime flags do not move primitive draws.

Only schema-v5 archives carry the new component truth; older archives are rejected, not silently zero-filled. Baseline seasonality has not been added, and seasonal inputs are not Fourier controls inserted into the baseline.


In [ ]:
manifest_rows = []
for source_id, shard, archive_path in (
    ("baseline8", corpus, BASELINE_PATH),
    ("mixed100", scaled_corpus, SCALED_PATH),
):
    for world_id in range(len(shard["cell_id"])):
        data = {"week": np.arange(shard["outcome_raw"].shape[1])}
        for role, key, prefix in (
            ("treatment", "treatment_raw", "C"),
            ("covariate", "covariates", "Z"),
        ):
            for slot in np.flatnonzero(shard[f"{role}_active_mask"][world_id]):
                data[f"{role}_{prefix}{slot + 1}"] = shard[key][world_id, :, slot]
        data["outcome_Y"] = shard["outcome_raw"][world_id]
        csv_path = RUN_DIR / f"{source_id}-world-{world_id}.csv"
        pd.DataFrame(data).to_csv(csv_path, index=False)
        manifest_rows.append(
            {
                "dataset": f"{source_id}:{world_id}",
                "source": source_id,
                "world_id": world_id,
                "generation_seed": GENERATION_SEED,
                "model_data": str(csv_path.relative_to(RUN_DIR)),
                "truth_source": archive_path.name,
            }
        )
for index, (name, bundle) in enumerate(atlas_paths.items()):
    manifest_rows.append(
        {
            "dataset": f"atlas:{name}:0",
            "source": f"atlas:{name}",
            "world_id": 0,
            "generation_seed": 100 + index,
            "model_data": str((Path(bundle) / "dataset.csv").relative_to(RUN_DIR)),
            "truth_source": str(Path(bundle).relative_to(RUN_DIR)),
        }
    )
manifest_rows.append(
    {
        "dataset": "replay:0",
        "source": "replay",
        "world_id": 0,
        "generation_seed": 27,
        "model_data": str((Path(rich_bundle) / "dataset.csv").relative_to(RUN_DIR)),
        "truth_source": str(Path(rich_bundle).relative_to(RUN_DIR)),
    }
)
manifest = pd.DataFrame(manifest_rows)
assert len(manifest) == 117
assert manifest["dataset"].is_unique and manifest["model_data"].is_unique
assert not manifest.duplicated(["source", "world_id"]).any()
manifest.to_csv(RUN_DIR / "manifest.csv", index=False)
display(manifest)
print("Dataset and report directory:", RUN_DIR.resolve())
print("Stored mixed shard:", CORPUS_PATH.resolve())
print("Stored eight-world baseline:", BASELINE_PATH.resolve())
print("Design screen:", report["decision"])
print(
    "Measured gaps:",
    report["inventory"]
    .loc[report["inventory"]["count"] == 0, ["role", "trajectory"]]
    .to_dict("records"),
)